# Probem a replicar el Blume-Capel

m(T) creua zero a ~29°C: vol dir que h_efectiu(T) creua zero a aquesta temperatura. La temperatura ambiental actua com un camp extern que empeny el sistema cap a comfort (h<0) per sota de 29°C i cap a discomfort (h>0) per sobre.


q(T) ≈ 0.72 constant: vol dir que D_efectiu no varia amb T. La fracció de població que està polaritzada (no neutral) és una propietat estable, independent de quanta calor faci. El que canvia amb T és cap a on es polaritza, no si es polaritza.


Això és el que dóna sentit físic al model. La temperatura ambiental no és la "temperatura termodinàmica" del sistema (que en Blume-Capel és el T_temp del coeficient de Boltzmann, i representaria la heterogeneïtat poblacional). La temperatura ambiental actua com a camp extern h. La heterogeneïtat poblacional juga el rol de T_temp, i el cost intrínsec de polaritzar-se juga el rol de D.

Si q és constant (com observes), i la mescla és vàlida, llavors el sistema està en una mateixa fase termodinàmica a tot el rang de T. No hi ha cap transició de fase en el sentit estricte. El que hi ha és un crossover suau del valor de h_efectiu, que arrossega m de negatiu a positiu sense canviar la fase subjacent.

El crossing α=0.5 ≈ 28.7°C i m=0 a ~29°C són la mateixa cosa expressada de dues maneres: és la temperatura on h_efectiu ≈ 0 i el sistema està en el punt de simetria entre les dues polaritzacions. No és una transició crítica, és el centre de l'eix d'asimetria.

La connexió amb la dinàmica a 30°C: la transició dinàmica R_D=1 NO és el mateix que m=0. R_D mesura el balanç de fluxos entre estats, no la composició de la distribució. La distinció en llenguatge Blume-Capel és:

m=0 a 28-29°C és la condició estàtica d'equilibri entre polaritzacions.
R_D=1 a 30°C és la condició dinàmica d'equilibri entre fluxos d'onset i recovery.

## Pseudo conclusió (idea)

Si el sistema fos en equilibri termodinàmic estricte (compleix detailed balance), els dos punts coincidirien. La separació de 1-2°C indica que el sistema està fora d'equilibri en aquest rang. Concretament: a 28-29°C la distribució està ja simètrica entre comfort i discomfort, però els fluxos dinàmics encara afavoreixen recovery. El sistema "encara s'està relaxant" cap al règim calent quan estàticament ja hi és proper.


Això és exactament la signatura d'un sistema amb mobilitats asimètriques: la geometria estàtica del sistema arriba a la simetria abans que la dinàmica perquè els canals de recovery i onset no responen amb la mateixa rapidesa a canvis de h_efectiu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path


# Put your notebook path here once (or set it via VS Code explorer)
NOTEBOOK_DIR = Path(r"C:\Users\labor\Documents\GitHub\TFM-Thermal-Walks\statistical_try")  # <-- change this

out_dir = NOTEBOOK_DIR / "figures"
out_dir.mkdir(parents=True, exist_ok=True)



df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")
print(df_votes.columns)



# Posem bé les dades

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from scipy import stats


# ============================================================
# CONFIG
# ============================================================


temp_corr_col = "<T-T_fixed+<T>>"
hdx_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"

# Order matters: S = -1, 0, +1
class_order = [
    "comfortable_side",   # S = -1
    "neutral",            # S =  0
    "uncomfortable_side", # S = +1
]


# ============================================================
# PREPARE DATA
# ============================================================

df = df_votes.copy()
df["walk"] = df["space_code"].astype(str).str[:2]

for c in [ temp_corr_col, hdx_col]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["T_rel_walk"] = (
    df[temp_corr_col]
    - df.groupby("walk")[temp_corr_col].transform("mean")
)


def comfort_side(cat):
    if cat in ["Very comfortable", "Comfortable", "Slightly comfortable"]:
        return "comfortable_side"
    elif cat == "Neutral":
        return "neutral"
    elif cat in ["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"]:
        return "uncomfortable_side"
    return np.nan


df["comfort_3"] = df[comfort_col].map(comfort_side)


# Funcions d'ajuda

In [ ]:
# ============================================================
# HELPERS
# ============================================================

def wilson_ci(k, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan

    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    phat = k / n

    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (
        z
        * np.sqrt((phat * (1 - phat) / n) + (z**2 / (4 * n**2)))
        / denom
    )

    return center - half, center + half


def predict_proba_fixed_order(model, x_grid, class_order):
    probs = model.predict_proba(x_grid.reshape(-1, 1))

    prob_df = pd.DataFrame(probs, columns=model.classes_)

    for c in class_order:
        if c not in prob_df.columns:
            prob_df[c] = np.nan

    return prob_df[class_order].to_numpy()


def crossing_x(x, y, target=0.0):
    """
    Estimate x where y crosses `target`.
    If no crossing exists, returns the x where |y - target| is smallest.
    """
    y = np.asarray(y) - target
    x = np.asarray(x)

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))]

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0

    return x0 - y0 * (x1 - x0) / (y1 - y0)


def compute_order_parameters(probs, class_order):
    """
    Given a probability array of shape (n_grid, 3) with columns in
    [comfortable_side, neutral, uncomfortable_side] order, returns:
      m = P(U) - P(C)
      q = P(U) + P(C) = 1 - P(N)
      delta = q - |m| = 2 * min(P(C), P(U))   [coexistence indicator]
      var_S = q - m**2                         [spin variance / susceptibility proxy]
    """
    iC = class_order.index("comfortable_side")
    iN = class_order.index("neutral")
    iU = class_order.index("uncomfortable_side")

    pC = probs[..., iC]
    pN = probs[..., iN]
    pU = probs[..., iU]

    m = pU - pC
    q = pU + pC
    delta = q - np.abs(m)
    var_S = q - m**2

    return m, q, delta, var_S


# Blume- Capel

In [ ]:

# ============================================================
# MAIN FUNCTION: BLUME-CAPEL ORDER PARAMETERS
# ==
# 
# ==========================================================

def fit_blume_capel_order_params(
    df_in,
    x_col,
    y_col="comfort_3",
    cluster_col="walk",
    n_boot=500,
    n_grid=500,
    n_bins=10,
    min_bin_n=10,
    random_state=123,
):
    rng = np.random.default_rng(random_state)

    sub = df_in[[x_col, y_col, cluster_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty:
        print(f"No data for {x_col}")
        return None

    print(f"\nPredictor: {x_col}")
    print(f"N = {len(sub)}")
    print(sub[y_col].value_counts().reindex(class_order))

    # ----------------------------
    # Fit original model
    # ----------------------------
    X = sub[[x_col]].to_numpy()
    y = sub[y_col].to_numpy()

    model = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=3000,
    )

    model.fit(X, y)

    x_min, x_max = np.nanpercentile(sub[x_col], [1, 99])
    x_grid = np.linspace(x_min, x_max, n_grid)

    probs_hat = predict_proba_fixed_order(model, x_grid, class_order)

    m_hat, q_hat, delta_hat, var_hat = compute_order_parameters(
        probs_hat, class_order
    )

    # ----------------------------
    # Crossings of interest
    # ----------------------------
    # 1) m = 0 (symmetry point)
    x_m0 = crossing_x(x_grid, m_hat, target=0.0)

    # 2) m = q (one extreme empty: P(C) = 0 from the comfort side)
    #    equivalent to delta = 0 from above
    x_m_eq_q = crossing_x(x_grid, m_hat - q_hat, target=0.0)

    # 3) m = -q (the other extreme empty: P(U) = 0)
    x_m_eq_neg_q = crossing_x(x_grid, m_hat + q_hat, target=0.0)

    # 4) Max of var_S = q - m^2 (susceptibility peak)
    i_chi_peak = int(np.nanargmax(var_hat))
    x_chi_peak = x_grid[i_chi_peak]

    # 5) Max of delta (maximum coexistence)
    i_delta_peak = int(np.nanargmax(delta_hat))
    x_delta_peak = x_grid[i_delta_peak]

    # ----------------------------
    # Bootstrap
    # ----------------------------
    boot_m = []
    boot_q = []
    boot_delta = []
    boot_var = []

    boot_x_m0 = []
    boot_x_m_eq_q = []
    boot_x_m_eq_neg_q = []
    boot_x_chi_peak = []
    boot_x_delta_peak = []

    clusters = sub[cluster_col].dropna().unique()

    attempts = 0

    while len(boot_m) < n_boot and attempts < n_boot * 10:
        attempts += 1

        sampled_clusters = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        boot_parts = []
        for cl in sampled_clusters:
            boot_parts.append(sub[sub[cluster_col] == cl])

        boot = pd.concat(boot_parts, axis=0)

        if boot[y_col].nunique() < len(class_order):
            continue

        Xb = boot[[x_col]].to_numpy()
        yb = boot[y_col].to_numpy()

        try:
            mb = LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=3000,
            )
            mb.fit(Xb, yb)

            pb = predict_proba_fixed_order(mb, x_grid, class_order)

            m_b, q_b, delta_b, var_b = compute_order_parameters(
                pb, class_order
            )

            boot_m.append(m_b)
            boot_q.append(q_b)
            boot_delta.append(delta_b)
            boot_var.append(var_b)

            boot_x_m0.append(crossing_x(x_grid, m_b, target=0.0))
            boot_x_m_eq_q.append(crossing_x(x_grid, m_b - q_b, target=0.0))
            boot_x_m_eq_neg_q.append(crossing_x(x_grid, m_b + q_b, target=0.0))
            boot_x_chi_peak.append(x_grid[int(np.nanargmax(var_b))])
            boot_x_delta_peak.append(x_grid[int(np.nanargmax(delta_b))])

        except Exception:
            continue

    boot_m = np.asarray(boot_m)
    boot_q = np.asarray(boot_q)
    boot_delta = np.asarray(boot_delta)
    boot_var = np.asarray(boot_var)

    boot_x_m0 = np.asarray(boot_x_m0)
    boot_x_m_eq_q = np.asarray(boot_x_m_eq_q)
    boot_x_m_eq_neg_q = np.asarray(boot_x_m_eq_neg_q)
    boot_x_chi_peak = np.asarray(boot_x_chi_peak)
    boot_x_delta_peak = np.asarray(boot_x_delta_peak)

    print(f"Bootstrap fits used: {len(boot_m)} / {n_boot}")

    # CIs
    m_low, m_high = np.nanpercentile(boot_m, [2.5, 97.5], axis=0)
    q_low, q_high = np.nanpercentile(boot_q, [2.5, 97.5], axis=0)
    delta_low, delta_high = np.nanpercentile(boot_delta, [2.5, 97.5], axis=0)
    var_low, var_high = np.nanpercentile(boot_var, [2.5, 97.5], axis=0)

    # Threshold CIs
    ci_m0 = np.nanpercentile(boot_x_m0, [2.5, 50, 97.5])
    ci_m_eq_q = np.nanpercentile(boot_x_m_eq_q, [2.5, 50, 97.5])
    ci_m_eq_neg_q = np.nanpercentile(boot_x_m_eq_neg_q, [2.5, 50, 97.5])
    ci_chi_peak = np.nanpercentile(boot_x_chi_peak, [2.5, 50, 97.5])
    ci_delta_peak = np.nanpercentile(boot_x_delta_peak, [2.5, 50, 97.5])

    print("\nCritical x-values with bootstrap 95% CI:")
    print(
        f"m = 0 (symmetry):           "
        f"{x_m0:.3f}  [{ci_m0[0]:.3f}, {ci_m0[2]:.3f}]"
    )
    print(
        f"m = +q (P(C) -> 0):         "
        f"{x_m_eq_q:.3f}  [{ci_m_eq_q[0]:.3f}, {ci_m_eq_q[2]:.3f}]"
    )
    print(
        f"m = -q (P(U) -> 0):         "
        f"{x_m_eq_neg_q:.3f}  [{ci_m_eq_neg_q[0]:.3f}, {ci_m_eq_neg_q[2]:.3f}]"
    )
    print(
        f"Var(S) peak (susceptib.):   "
        f"{x_chi_peak:.3f}  [{ci_chi_peak[0]:.3f}, {ci_chi_peak[2]:.3f}]"
    )
    print(
        f"Delta peak (coexistence):   "
        f"{x_delta_peak:.3f}  [{ci_delta_peak[0]:.3f}, {ci_delta_peak[2]:.3f}]"
    )

    # ----------------------------
    # Empirical binned order params with Wilson-style propagation
    # ----------------------------
    sub["x_bin"] = pd.qcut(
        sub[x_col],
        q=n_bins,
        duplicates="drop",
    )

    rows = []

    for b, g in sub.groupby("x_bin", observed=False):
        n = len(g)

        if n < min_bin_n:
            continue

        kC = (g[y_col] == "comfortable_side").sum()
        kN = (g[y_col] == "neutral").sum()
        kU = (g[y_col] == "uncomfortable_side").sum()

        pC = kC / n
        pN = kN / n
        pU = kU / n

        m_emp = pU - pC
        q_emp = pU + pC
        delta_emp = q_emp - abs(m_emp)
        var_emp = q_emp - m_emp**2

        # CIs for m and q via Wilson on the contributing classes
        loC, hiC = wilson_ci(kC, n)
        loU, hiU = wilson_ci(kU, n)
        loN, hiN = wilson_ci(kN, n)

        # Conservative band for m
        m_lo = loU - hiC
        m_hi = hiU - loC

        # Conservative band for q (sum of independent CIs)
        q_lo = loU + loC
        q_hi = hiU + hiC

        rows.append({
            "x_mean": g[x_col].mean(),
            "n": n,
            "pC": pC, "pN": pN, "pU": pU,
            "m": m_emp, "m_lo": m_lo, "m_hi": m_hi,
            "q": q_emp, "q_lo": q_lo, "q_hi": q_hi,
            "delta": delta_emp,
            "var_S": var_emp,
        })

    bin_df = pd.DataFrame(rows)

    # ============================================================
    # PLOTS
    # ============================================================

    # ------------------------------------------------------------
    # Plot 1: m(T) and q(T) joint
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 5.5))

    ax.fill_between(x_grid, m_low, m_high, alpha=0.20, color="C0")
    ax.plot(x_grid, m_hat, color="C0", linewidth=2,
            label="m = P(U) - P(C)")

    ax.fill_between(x_grid, q_low, q_high, alpha=0.20, color="C3")
    ax.plot(x_grid, q_hat, color="C3", linewidth=2,
            label="q = P(U) + P(C) = 1 - P(N)")

    if not bin_df.empty:
        ax.errorbar(
            bin_df["x_mean"], bin_df["m"],
            yerr=[bin_df["m"] - bin_df["m_lo"], bin_df["m_hi"] - bin_df["m"]],
            fmt="o", color="C0", alpha=0.65, capsize=3,
        )
        ax.errorbar(
            bin_df["x_mean"], bin_df["q"],
            yerr=[bin_df["q"] - bin_df["q_lo"], bin_df["q_hi"] - bin_df["q"]],
            fmt="s", color="C3", alpha=0.65, capsize=3,
        )

    ax.axhline(0, linestyle="--", color="gray", alpha=0.6)

    # Mark crossings
    ax.axvline(x_m0, linestyle="--", color="C0", alpha=0.8,
               label=f"m=0 at {x_m0:.2f}")
    ax.axvspan(ci_m0[0], ci_m0[2], color="C0", alpha=0.08)

    # ax.axvline(x_chi_peak, linestyle=":", color="purple", alpha=0.8,
    #            label=f"Var(S) peak at {x_chi_peak:.2f}")
    # ax.axvspan(ci_chi_peak[0], ci_chi_peak[2], color="purple", alpha=0.08)

    ax.set_xlabel(x_col)
    ax.set_ylabel("Order parameters")
    ax.set_title(
        f"Blume-Capel order parameters vs {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()



    # ------------------------------------------------------------
    # Plot 3: Delta(T) = q - |m|, coexistence indicator
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 4.8))

    ax.fill_between(x_grid, delta_low, delta_high, alpha=0.20, color="C2")
    ax.plot(x_grid, delta_hat, color="C2", linewidth=2,
            label=r"$\Delta = q - |m| = 2\min(P(C), P(U))$")

    if not bin_df.empty:
        ax.scatter(bin_df["x_mean"], bin_df["delta"],
                   color="C2", edgecolor="black", s=40, alpha=0.75,
                   label="Empirical bins")

    ax.axvline(x_delta_peak, linestyle="--", color="C2", alpha=0.8,
               label=f"Delta peak at {x_delta_peak:.2f}")
    ax.axvspan(ci_delta_peak[0], ci_delta_peak[2], color="C2", alpha=0.08)

    ax.axvline(x_m0, linestyle=":", color="C0", alpha=0.6,
               label=f"m=0 at {x_m0:.2f}  (for reference)")

    ax.set_xlabel(x_col)
    ax.set_ylabel(r"$\Delta(T)$")
    
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc="best")
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Plot 4: Var(S) = q - m^2, susceptibility proxy
    # ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 4.8))

    ax.fill_between(x_grid, var_low, var_high, alpha=0.20, color="purple")
    ax.plot(x_grid, var_hat, color="purple", linewidth=2,
            label=r"Var(S) = q - m$^2$")

    if not bin_df.empty:
        ax.scatter(bin_df["x_mean"], bin_df["var_S"],
                   color="purple", edgecolor="black", s=40, alpha=0.75,
                   label="Empirical bins")

    ax.axvline(x_chi_peak, linestyle="--", color="purple", alpha=0.8,
               label=f"Peak at {x_chi_peak:.2f}")
    ax.axvspan(ci_chi_peak[0], ci_chi_peak[2], color="purple", alpha=0.08)

    ax.axvline(x_m0, linestyle=":", color="C0", alpha=0.6,
               label=f"m=0 at {x_m0:.2f}  (for reference)")

    ax.set_xlabel(x_col)
    ax.set_ylabel("Var(S)")
    ax.set_title(
        f"Spin variance / susceptibility proxy vs {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc="best")
    plt.tight_layout()
    plt.show()


    # ============================================================
    # DIAGNOSTIC CHECKS
    # ============================================================

    print("\n" + "=" * 55)
    print("DIAGNOSTIC REPORT")
    print("=" * 55)

    # ----------------------------
    # Check A: Is q(T) statistically constant?
    # ----------------------------
    q_at_low  = boot_q[:, 0]
    q_at_high = boot_q[:, -1]

    diff_q = q_at_high - q_at_low
    diff_mean = np.mean(diff_q)
    diff_ci   = np.nanpercentile(diff_q, [2.5, 97.5])

    print("\n[A] Is q(T) constant across the temperature range?")
    print(
        f"    q(T_high) - q(T_low) = {diff_mean:.3f} "
        f"[{diff_ci[0]:.3f}, {diff_ci[1]:.3f}]  (95% bootstrap CI)"
    )
    if diff_ci[0] <= 0 <= diff_ci[1]:
        print("    --> CI includes 0: q is statistically CONSTANT.")
        print("        Interpretation: T shifts m but not q.")
        print("        D_eff is approximately stable; h_eff(T) drives the transition.")
    else:
        sign = "increases" if diff_mean > 0 else "decreases"
        print(f"    --> CI excludes 0: q significantly {sign} with T.")
        print("        Interpretation: T also modulates population activation (D_eff varies).")

    # ----------------------------
    # Check B: Is the Var(S) peak driven by q's slope or genuine asymmetry?
    # ----------------------------
    print("\n[B] Is the Var(S) peak displaced by q's slope or genuine asymmetry?")

    # Normalised susceptibility: chi_tilde = Var(S) / q = 1 - m^2/q
    # Avoid division by zero
    q_safe = np.where(q_hat > 1e-6, q_hat, np.nan)
    chi_tilde = var_hat / q_safe

    i_chi_tilde_peak = int(np.nanargmax(chi_tilde))
    x_chi_tilde_peak = x_grid[i_chi_tilde_peak]

    # Bootstrap version
    boot_x_chi_tilde_peak = []
    for bq, bv in zip(boot_q, boot_var):
        bq_safe = np.where(bq > 1e-6, bq, np.nan)
        ct = bv / bq_safe
        boot_x_chi_tilde_peak.append(x_grid[int(np.nanargmax(ct))])

    boot_x_chi_tilde_peak = np.asarray(boot_x_chi_tilde_peak)
    ci_chi_tilde_peak = np.nanpercentile(boot_x_chi_tilde_peak, [2.5, 50, 97.5])

    displacement = x_chi_peak - x_chi_tilde_peak

    print(
        f"    Var(S) peak:            {x_chi_peak:.3f}  "
        f"[{ci_chi_peak[0]:.3f}, {ci_chi_peak[2]:.3f}]"
    )
    print(
        f"    Var(S)/q peak:          {x_chi_tilde_peak:.3f}  "
        f"[{ci_chi_tilde_peak[0]:.3f}, {ci_chi_tilde_peak[2]:.3f}]"
    )
    print(f"    m=0 crossing:           {x_m0:.3f}  [{ci_m0[0]:.3f}, {ci_m0[2]:.3f}]")
    print(f"    Displacement Var(S) - Var(S)/q: {displacement:.3f}")

    if abs(x_chi_tilde_peak - x_m0) < abs(x_chi_peak - x_m0):
        print(
            "    --> Normalising by q moves the peak closer to m=0."
        )
        print(
            "        The displacement of Var(S) is partly driven by q's slope."
        )
        if abs(x_chi_tilde_peak - x_m0) < 0.5:
            print(
                "        After normalisation, peak is close to m=0: "
                "asymmetry is weak, near mean-field symmetric."
            )
        else:
            print(
                "        Residual displacement remains: genuine asymmetry in h_eff(T)."
            )
    else:
        print(
            "    --> Normalising by q does NOT move the peak toward m=0."
        )
        print(
            "        The asymmetry is genuine, not an artefact of q's slope."
        )

    # Also plot chi_tilde for visual inspection
    fig, ax = plt.subplots(figsize=(9, 4.5))

    boot_chi_tilde_arr = np.asarray([
        bv / np.where(bq > 1e-6, bq, np.nan)
        for bq, bv in zip(boot_q, boot_var)
    ])
    ct_low  = np.nanpercentile(boot_chi_tilde_arr, 2.5,  axis=0)
    ct_high = np.nanpercentile(boot_chi_tilde_arr, 97.5, axis=0)

    ax.fill_between(x_grid, ct_low, ct_high, alpha=0.20, color="darkorchid")
    ax.plot(x_grid, chi_tilde, color="darkorchid", linewidth=2,
            label=r"$\tilde{\chi}$ = Var(S) / q")

    if not bin_df.empty:
        chi_tilde_bin = bin_df["var_S"] / bin_df["q"].replace(0, np.nan)
        ax.scatter(bin_df["x_mean"], chi_tilde_bin,
                   color="darkorchid", edgecolor="black", s=40, alpha=0.75,
                   label="Empirical bins")

    ax.axvline(x_chi_tilde_peak, linestyle="--", color="darkorchid", alpha=0.8,
               label=f"Var(S)/q peak at {x_chi_tilde_peak:.2f}")
    ax.axvspan(ci_chi_tilde_peak[0], ci_chi_tilde_peak[2],
               color="darkorchid", alpha=0.08)

    ax.axvline(x_m0, linestyle=":", color="C0", alpha=0.7,
               label=f"m=0 at {x_m0:.2f}  (reference)")
    ax.axvline(x_chi_peak, linestyle=":", color="purple", alpha=0.5,
               label=f"Var(S) peak at {x_chi_peak:.2f}  (reference)")

    ax.set_xlabel(x_col)
    ax.set_ylabel(r"$\tilde{\chi}$ = Var(S) / q")
    ax.set_title(
        f"[Check B] Normalised susceptibility vs {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Check C: Does q vary across walks?
    # ----------------------------
    print("\n[C] Walk-stratified q: does q vary across walks?")

    walk_q = (
        df_in[df_in[y_col].isin(class_order)]
        .groupby(cluster_col)[y_col]
        .apply(lambda s: 1 - (s == "neutral").mean())
        .dropna()
    )

    q_mean   = walk_q.mean()
    q_std    = walk_q.std()
    q_min    = walk_q.min()
    q_max    = walk_q.max()
    q_iqr    = walk_q.quantile(0.75) - walk_q.quantile(0.25)

    print(f"    N walks = {len(walk_q)}")
    print(f"    q per walk:  mean={q_mean:.3f}  std={q_std:.3f}  "
          f"min={q_min:.3f}  max={q_max:.3f}  IQR={q_iqr:.3f}")

    if q_std / q_mean > 0.15:
        print(
            "    --> High walk-level variability (CV > 15%)."
        )
        print(
            "        The apparent constancy of q(T) may reflect averaging over"
        )
        print(
            "        heterogeneous walks, not a genuine physical property."
        )
        print(
            "        Consider checking if q correlates with HDX or city."
        )
    else:
        print(
            "    --> Low walk-level variability (CV <= 15%)."
        )
        print(
            "        q is consistently high across walks: population activation"
        )
        print(
            "        is a robust property of the dataset, not a mixing artefact."
        )

    # Distribution plot
    fig, ax = plt.subplots(figsize=(8, 4))

    ax.hist(walk_q.values, bins=15, color="C3", alpha=0.7, edgecolor="white")
    ax.axvline(q_mean, linestyle="--", color="black",
               label=f"Mean = {q_mean:.3f}")
    ax.axvline(q_mean - q_std, linestyle=":", color="gray",
               label=f"±1 std ({q_std:.3f})")
    ax.axvline(q_mean + q_std, linestyle=":", color="gray")
    ax.axvline(q_hat.mean(), linestyle="-", color="C0", alpha=0.7,
               label=f"Model mean q = {q_hat.mean():.3f}")

    ax.set_xlabel("q per walk  [= 1 - P(neutral)]")
    ax.set_ylabel("Number of walks")
    ax.set_title("[Check C] Walk-level distribution of q")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 55)

    # ============================================================
    # RETURN
    # ============================================================
    result = {
        "predictor": x_col,
        "n": len(sub),
        "model": model,
        "x_grid": x_grid,
        "probs_hat": probs_hat,
        "m_hat": m_hat,
        "q_hat": q_hat,
        "delta_hat": delta_hat,
        "var_hat": var_hat,
        "bin_df": bin_df,
        "boot_m": boot_m,
        "boot_q": boot_q,
        "boot_delta": boot_delta,
        "boot_var": boot_var,
        "x_m0": x_m0, "ci_m0": ci_m0,
        "x_m_eq_q": x_m_eq_q, "ci_m_eq_q": ci_m_eq_q,
        "x_m_eq_neg_q": x_m_eq_neg_q, "ci_m_eq_neg_q": ci_m_eq_neg_q,
        "x_chi_peak": x_chi_peak, "ci_chi_peak": ci_chi_peak,
        "x_delta_peak": x_delta_peak, "ci_delta_peak": ci_delta_peak,
    }

    return result


# ============================================================
# RUN
# ============================================================


# Example: with corrected temperature
res_Tcorr = fit_blume_capel_order_params(df, x_col=temp_corr_col)

# Example: with HDX
res_HDX = fit_blume_capel_order_params(df, x_col=hdx_col)

# Tractament d'outliers

In [ ]:
# Afegeix això al teu notebook per identificar els outliers
walk_q = (
    df[df["comfort_3"].isin(class_order)]
    .groupby("walk")["comfort_3"]
    .apply(lambda s: 1 - (s == "neutral").mean())
    .sort_values()
)

# Walks amb q baix (molts neutrals)
print("Walks with lowest q (most neutral responses):")
print(walk_q.head(5))

# Enriqueix amb temperatura mitjana del walk
walk_info = df.groupby("walk").agg(
    q_walk=("comfort_3", lambda s: 1 - (s == "neutral").mean()),
    T_mean=(temp_corr_col, "mean"),
    n=("comfort_3", "count"),
).sort_values("q_walk")

print("\nWalk info sorted by q:")
print(walk_info.head(10).to_string())

In [ ]:
# ============================================================
# WALK OUTLIER DEEP DIVE
# ============================================================

outlier_walks = ["2S", "1C", "5G", "1X", "1I"]

# Variables que vols inspeccionar per walk
# Ajusta els noms de columna al teu dataset
context_cols = [
    temp_corr_col,          # temperatura
    hdx_col,                # HDX
    "sun_exposure",         # si tens exposició solar (o similar)
    "wind_speed",           # vent
    "city",                 # ciutat (si tens columna)
    "comfort_3",            # per calcular distribució d'estats
]

available_cols = [c for c in context_cols if c in df.columns]

print("=" * 60)
print("WALK OUTLIER ANALYSIS")
print("=" * 60)

for w in outlier_walks:
    sub = df[df["walk"] == w].copy()
    sub_valid = sub[sub["comfort_3"].isin(class_order)]

    n_total = len(sub_valid)
    pC = (sub_valid["comfort_3"] == "comfortable_side").mean()
    pN = (sub_valid["comfort_3"] == "neutral").mean()
    pU = (sub_valid["comfort_3"] == "uncomfortable_side").mean()
    q_w = pC + pU
    m_w = pU - pC

    print(f"\nWalk: {w}")
    print(f"  n = {n_total}")
    print(f"  T_mean = {sub[temp_corr_col].mean():.2f}°C  "
          f"  T_std = {sub[temp_corr_col].std():.2f}°C")

    if hdx_col in df.columns:
        print(f"  HDX_mean = {sub[hdx_col].mean():.2f}  "
              f"  HDX_std = {sub[hdx_col].std():.2f}")

    print(f"  P(C) = {pC:.3f}  |  P(N) = {pN:.3f}  |  P(U) = {pU:.3f}")
    print(f"  m = {m_w:+.3f}  |  q = {q_w:.3f}")

    # Mirem si la ciutat és identificable pel prefix
    city_prefix = w[0]
    print(f"  City prefix: {city_prefix}")

    # Distribució de TSV si la tens
    if "thermal_sensation" in df.columns:
        tsv_mean = sub["thermal_sensation"].map({
            "Very cold": -3, "Cold": -2, "Cool": -1,
            "Neutral": 0,
            "Warm": 1, "Hot": 2, "Very hot": 3
        }).mean()
        if not np.isnan(tsv_mean):
            print(f"  TSV_mean (numeric) = {tsv_mean:.2f}")

print("\n" + "=" * 60)

# -------------------------------------------------------
# Scatter: q vs T per walk, destacant outliers
# -------------------------------------------------------
walk_info = df[df["comfort_3"].isin(class_order)].groupby("walk").agg(
    q_walk=("comfort_3", lambda s: 1 - (s == "neutral").mean()),
    m_walk=("comfort_3", lambda s:
            (s == "uncomfortable_side").mean() -
            (s == "comfortable_side").mean()),
    T_mean=(temp_corr_col, "mean"),
    n=("comfort_3", "count"),
).reset_index()

if hdx_col in df.columns:
    hdx_by_walk = df.groupby("walk")[hdx_col].mean().rename("HDX_mean")
    walk_info = walk_info.join(hdx_by_walk, on="walk")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: q vs T ---
ax = axes[0]

is_outlier = walk_info["walk"].isin(outlier_walks)

ax.scatter(
    walk_info.loc[~is_outlier, "T_mean"],
    walk_info.loc[~is_outlier, "q_walk"],
    s=walk_info.loc[~is_outlier, "n"] * 0.8,
    color="C3", alpha=0.55, edgecolor="white",
    label="Regular walks"
)
ax.scatter(
    walk_info.loc[is_outlier, "T_mean"],
    walk_info.loc[is_outlier, "q_walk"],
    s=walk_info.loc[is_outlier, "n"] * 0.8,
    color="C0", alpha=0.85, edgecolor="black", linewidth=1.2,
    label="Low-q outliers", zorder=5
)

for _, row in walk_info[is_outlier].iterrows():
    ax.annotate(
        row["walk"],
        (row["T_mean"], row["q_walk"]),
        textcoords="offset points", xytext=(6, 3),
        fontsize=8, color="C0"
    )

ax.axhline(walk_info["q_walk"].mean(), linestyle="--",
           color="gray", alpha=0.6, label=f"Mean q = {walk_info['q_walk'].mean():.3f}")
ax.set_xlabel("T_mean per walk (°C)")
ax.set_ylabel("q per walk  [= 1 - P(neutral)]")
ax.set_title("Walk-level q vs mean temperature")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Right: q vs HDX (if available) ---
ax = axes[1]

if "HDX_mean" in walk_info.columns:
    ax.scatter(
        walk_info.loc[~is_outlier, "HDX_mean"],
        walk_info.loc[~is_outlier, "q_walk"],
        s=walk_info.loc[~is_outlier, "n"] * 0.8,
        color="C3", alpha=0.55, edgecolor="white",
        label="Regular walks"
    )
    ax.scatter(
        walk_info.loc[is_outlier, "HDX_mean"],
        walk_info.loc[is_outlier, "q_walk"],
        s=walk_info.loc[is_outlier, "n"] * 0.8,
        color="C0", alpha=0.85, edgecolor="black", linewidth=1.2,
        label="Low-q outliers", zorder=5
    )
    for _, row in walk_info[is_outlier].iterrows():
        ax.annotate(
            row["walk"],
            (row["HDX_mean"], row["q_walk"]),
            textcoords="offset points", xytext=(6, 3),
            fontsize=8, color="C0"
        )

    # Afegim regressió lineal simple per veure tendència
    from scipy.stats import linregress
    mask_valid = walk_info["HDX_mean"].notna() & walk_info["q_walk"].notna()
    slope, intercept, r, p, _ = linregress(
        walk_info.loc[mask_valid, "HDX_mean"],
        walk_info.loc[mask_valid, "q_walk"]
    )
    x_hdx = np.linspace(
        walk_info["HDX_mean"].min(),
        walk_info["HDX_mean"].max(), 100
    )
    ax.plot(x_hdx, intercept + slope * x_hdx,
            linestyle="--", color="gray", alpha=0.7,
            label=f"OLS: r={r:.2f}, p={p:.3f}")

    ax.axhline(walk_info["q_walk"].mean(), linestyle=":",
               color="gray", alpha=0.5)
    ax.set_xlabel("HDX_mean per walk")
    ax.set_ylabel("q per walk  [= 1 - P(neutral)]")
    ax.set_title("Walk-level q vs mean HDX\n(size = n participants)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "HDX not available",
                 ha="center", va="center", transform=axes[1].transAxes)
    axes[1].set_title("q vs HDX (not available)")

plt.tight_layout()
plt.show()

In [ ]:
# Taula resum dels outliers per la tesi
import pandas as pd

outlier_summary = pd.DataFrame({
    "Walk": ["2S", "1C", "5G", "1X", "1I"],
    "T_mean": [27.66, 23.89, 29.24, 26.69, 29.96],
    "HDX_mean": [33.54, 29.00, 40.25, 30.53, 38.99],
    "TSV_mean": [0.50, 0.74, 1.04, 0.44, 0.80],
    "P(N)": [0.560, 0.531, 0.514, 0.422, 0.420],
    "q": [0.440, 0.469, 0.486, 0.578, 0.580],
    "m": [-0.200, -0.265, +0.086, -0.089, -0.060],
    "Mechanism": [
        "Microclimate",
        "Low T (genuine)",
        "Behavioural/demographic",
        "Low T + isothermal",
        "Behavioural/demographic",
    ]
}).set_index("Walk")

print(outlier_summary.to_string())

# Ho probem amb les distribucions originals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats



# ============================================================
# HELPERS
# ============================================================

def wilson_ci(k, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan

    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    phat = k / n

    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (
        z
        * np.sqrt((phat * (1 - phat) / n) + (z**2 / (4 * n**2)))
        / denom
    )

    return center - half, center + half


def linear_interp_crossing(x, y, target=0.0):
    """Linear interpolation of crossing point y(x) = target between bins."""
    y = np.asarray(y) - target
    x = np.asarray(x)

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))], None

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0, (x0, x1)

    crossing = x0 - y0 * (x1 - x0) / (y1 - y0)

    return crossing, (min(x0, x1), max(x0, x1))


# ============================================================
# MAIN FUNCTION: EMPIRICAL BLUME-CAPEL ORDER PARAMETERS
# ============================================================

def empirical_blume_capel(
    df_in,
    x_col,
    y_col="comfort_3",
    cluster_col="walk",
    class_order=None,
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
    n_boot=500,
    random_state=123,
    plot=True,
):
    """
    Compute Blume-Capel order parameters (m, q, Delta, Var(S), tilde_chi)
    directly from empirical binned probabilities, without any parametric fit.

    Uncertainty from two sources:
      - Within-bin Wilson CIs on the underlying class probabilities
      - Walk-level cluster bootstrap on bin assignments
    """

    if class_order is None:
        class_order = ["comfortable_side", "neutral", "uncomfortable_side"]

    rng = np.random.default_rng(random_state)

    sub = df_in[[x_col, y_col, cluster_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty:
        print(f"No data for {x_col}")
        return None

    print(f"\nPredictor: {x_col}")
    print(f"N = {len(sub)}")
    print(sub[y_col].value_counts().reindex(class_order))

    # ----------------------------------------------------
    # Bin assignment (on the FULL sample, used for ALL replicas)
    # ----------------------------------------------------
    if binning == "quantile":
        sub["x_bin"] = pd.qcut(sub[x_col], q=n_bins, duplicates="drop")
    elif binning == "equal_width":
        sub["x_bin"] = pd.cut(sub[x_col], bins=n_bins, include_lowest=True)
    else:
        raise ValueError("binning must be 'quantile' or 'equal_width'")

    # ----------------------------------------------------
    # ORIGINAL ESTIMATE: empirical bins with Wilson CI
    # ----------------------------------------------------
    rows = []

    for b, g in sub.groupby("x_bin", observed=False):
        n = len(g)

        if n < min_bin_n:
            continue

        kC = (g[y_col] == "comfortable_side").sum()
        kN = (g[y_col] == "neutral").sum()
        kU = (g[y_col] == "uncomfortable_side").sum()

        pC = kC / n
        pN = kN / n
        pU = kU / n

        loC, hiC = wilson_ci(kC, n)
        loN, hiN = wilson_ci(kN, n)
        loU, hiU = wilson_ci(kU, n)

        # Order parameters
        m  = pU - pC
        q  = pU + pC
        delta = q - abs(m)
        var_S = q - m**2
        chi_tilde = var_S / q if q > 1e-6 else np.nan

        # Wilson-based intervals propagated naively (conservative)
        m_lo = loU - hiC
        m_hi = hiU - loC
        q_lo = max(0.0, loU + loC)
        q_hi = min(1.0, hiU + hiC)

        rows.append({
            "bin": b,
            "bin_left":  b.left  if hasattr(b, "left")  else np.nan,
            "bin_right": b.right if hasattr(b, "right") else np.nan,
            "x_mean": g[x_col].mean(),
            "n": n,
            "pC": pC, "pN": pN, "pU": pU,
            "pC_lo": loC, "pC_hi": hiC,
            "pN_lo": loN, "pN_hi": hiN,
            "pU_lo": loU, "pU_hi": hiU,
            "m": m, "m_lo_wilson": m_lo, "m_hi_wilson": m_hi,
            "q": q, "q_lo_wilson": q_lo, "q_hi_wilson": q_hi,
            "delta": delta,
            "var_S": var_S,
            "chi_tilde": chi_tilde,
        })

    bin_df = pd.DataFrame(rows).sort_values("x_mean").reset_index(drop=True)

    if bin_df.empty:
        print("No valid bins after min_bin_n filtering.")
        return None

    # ----------------------------------------------------
    # WALK-LEVEL BOOTSTRAP on the SAME bin edges
    # ----------------------------------------------------
    # We resample walks, then recompute per-bin probabilities
    bin_index = bin_df["bin"].tolist()
    n_bins_kept = len(bin_index)

    clusters = sub[cluster_col].dropna().unique()

    boot_m = np.full((n_boot, n_bins_kept), np.nan)
    boot_q = np.full((n_boot, n_bins_kept), np.nan)
    boot_delta = np.full((n_boot, n_bins_kept), np.nan)
    boot_var = np.full((n_boot, n_bins_kept), np.nan)
    boot_chi = np.full((n_boot, n_bins_kept), np.nan)

    boot_x_m0 = np.full(n_boot, np.nan)
    boot_x_delta_peak = np.full(n_boot, np.nan)
    boot_x_chi_peak = np.full(n_boot, np.nan)
    boot_x_var_peak = np.full(n_boot, np.nan)

    for b in range(n_boot):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        boot = pd.concat(
            [sub[sub[cluster_col] == cl] for cl in sampled],
            axis=0,
        )

        # Use the SAME bin edges to keep comparability
        # Iterate bins by their categorical label
        for j, b_label in enumerate(bin_index):
            g = boot[boot["x_bin"] == b_label]
            n_g = len(g)

            if n_g < 1:
                continue

            kC = (g[y_col] == "comfortable_side").sum()
            kU = (g[y_col] == "uncomfortable_side").sum()

            pC = kC / n_g
            pU = kU / n_g

            m_b = pU - pC
            q_b = pU + pC
            d_b = q_b - abs(m_b)
            v_b = q_b - m_b**2
            c_b = v_b / q_b if q_b > 1e-6 else np.nan

            boot_m[b, j] = m_b
            boot_q[b, j] = q_b
            boot_delta[b, j] = d_b
            boot_var[b, j] = v_b
            boot_chi[b, j] = c_b

        # Crossings/peaks per replica
        x_centers = bin_df["x_mean"].values

        if np.isfinite(boot_m[b]).sum() >= 2:
            x_m0, _ = linear_interp_crossing(x_centers, boot_m[b], target=0.0)
            boot_x_m0[b] = x_m0

        if np.isfinite(boot_delta[b]).any():
            boot_x_delta_peak[b] = x_centers[np.nanargmax(boot_delta[b])]
        if np.isfinite(boot_var[b]).any():
            boot_x_var_peak[b] = x_centers[np.nanargmax(boot_var[b])]
        if np.isfinite(boot_chi[b]).any():
            boot_x_chi_peak[b] = x_centers[np.nanargmax(boot_chi[b])]

    # ----------------------------------------------------
    # CIs per bin from bootstrap
    # ----------------------------------------------------
    bin_df["m_lo_boot"]  = np.nanpercentile(boot_m, 2.5, axis=0)
    bin_df["m_hi_boot"]  = np.nanpercentile(boot_m, 97.5, axis=0)
    bin_df["q_lo_boot"]  = np.nanpercentile(boot_q, 2.5, axis=0)
    bin_df["q_hi_boot"]  = np.nanpercentile(boot_q, 97.5, axis=0)
    bin_df["delta_lo_boot"] = np.nanpercentile(boot_delta, 2.5, axis=0)
    bin_df["delta_hi_boot"] = np.nanpercentile(boot_delta, 97.5, axis=0)
    bin_df["var_lo_boot"]   = np.nanpercentile(boot_var, 2.5, axis=0)
    bin_df["var_hi_boot"]   = np.nanpercentile(boot_var, 97.5, axis=0)
    bin_df["chi_lo_boot"]   = np.nanpercentile(boot_chi, 2.5, axis=0)
    bin_df["chi_hi_boot"]   = np.nanpercentile(boot_chi, 97.5, axis=0)

    # ----------------------------------------------------
    # Critical points from the ORIGINAL estimate
    # ----------------------------------------------------
    x_centers = bin_df["x_mean"].values

    x_m0, m0_interval = linear_interp_crossing(
        x_centers, bin_df["m"].values, target=0.0
    )

    i_delta_peak = int(np.nanargmax(bin_df["delta"].values))
    x_delta_peak = x_centers[i_delta_peak]

    i_var_peak = int(np.nanargmax(bin_df["var_S"].values))
    x_var_peak = x_centers[i_var_peak]

    i_chi_peak = int(np.nanargmax(bin_df["chi_tilde"].values))
    x_chi_peak = x_centers[i_chi_peak]

    # Després de calcular x_chi_peak
    chi_values = bin_df["chi_tilde"].values
    threshold = 0.95  # o el percentil 95 dels valors

    above = chi_values >= threshold
    if above.any():
        idx_above = np.where(above)[0]
        critical_window = (
            bin_df["x_mean"].iloc[idx_above[0]],
            bin_df["x_mean"].iloc[idx_above[-1]],
        )
        print(
            f"  Critical window (chi_tilde >= {threshold}): "
            f"[{critical_window[0]:.2f}, {critical_window[1]:.2f}]  "
            f"width = {critical_window[1] - critical_window[0]:.2f}°C"
        )
    # Bootstrap CIs of critical points
    ci_m0  = np.nanpercentile(boot_x_m0,  [2.5, 50, 97.5])
    ci_dpk = np.nanpercentile(boot_x_delta_peak, [2.5, 50, 97.5])
    ci_vpk = np.nanpercentile(boot_x_var_peak,   [2.5, 50, 97.5])
    ci_cpk = np.nanpercentile(boot_x_chi_peak,   [2.5, 50, 97.5])

    # Wilson-CI overlap area for P(U) and P(C): defines the "uncertainty area"
    # around the crossing m=0 directly from the data
    ci_overlap = (
        (bin_df["pC_lo"] <= bin_df["pU_hi"])
        &
        (bin_df["pU_lo"] <= bin_df["pC_hi"])
    )

    overlap_bins = bin_df[ci_overlap]
    if len(overlap_bins) > 0:
        wilson_overlap_interval = (
            overlap_bins["bin_left"].min(),
            overlap_bins["bin_right"].max(),
        )
    else:
        wilson_overlap_interval = None

    # ----------------------------------------------------
    # Print summary
    # ----------------------------------------------------
    print("\nEmpirical Blume-Capel critical points:")
    print(
        f"  m = 0:           {x_m0:.3f}   "
        f"bootstrap 95% CI [{ci_m0[0]:.3f}, {ci_m0[2]:.3f}]"
    )
    if wilson_overlap_interval is not None:
        print(
            f"  Wilson-CI overlap area for m=0: "
            f"[{wilson_overlap_interval[0]:.3f}, {wilson_overlap_interval[1]:.3f}]"
        )
    print(
        f"  Delta peak:      {x_delta_peak:.3f}   "
        f"bootstrap 95% CI [{ci_dpk[0]:.3f}, {ci_dpk[2]:.3f}]"
    )
    print(
        f"  Var(S) peak:     {x_var_peak:.3f}   "
        f"bootstrap 95% CI [{ci_vpk[0]:.3f}, {ci_vpk[2]:.3f}]"
    )
    print(
        f"  Var(S)/q peak:   {x_chi_peak:.3f}   "
        f"bootstrap 95% CI [{ci_cpk[0]:.3f}, {ci_cpk[2]:.3f}]"
    )

    # ============================================================
    # PLOTS
    # ============================================================
    if plot:
        # --------------------------------------------------------
        # Plot 1: m and q (with Wilson AND bootstrap bands)
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(10, 5.5))

        x = bin_df["x_mean"].values

        # m: bootstrap band + Wilson errorbars
        ax.fill_between(
            x, bin_df["m_lo_boot"], bin_df["m_hi_boot"],
            color="C0", alpha=0.20, label="m: walk bootstrap 95% CI"
        )
        ax.errorbar(
            x, bin_df["m"],
            yerr=[bin_df["m"] - bin_df["m_lo_wilson"],
                  bin_df["m_hi_wilson"] - bin_df["m"]],
            fmt="o", color="C0", capsize=3, alpha=0.85,
            label="m (Wilson errorbar)"
        )

        # q: bootstrap band + Wilson errorbars
        ax.fill_between(
            x, bin_df["q_lo_boot"], bin_df["q_hi_boot"],
            color="C3", alpha=0.20, label="q: walk bootstrap 95% CI"
        )
        ax.errorbar(
            x, bin_df["q"],
            yerr=[bin_df["q"] - bin_df["q_lo_wilson"],
                  bin_df["q_hi_wilson"] - bin_df["q"]],
            fmt="s", color="C3", capsize=3, alpha=0.85,
            label="q (Wilson errorbar)"
        )

        ax.axhline(0, linestyle="--", color="gray", alpha=0.6)

        # m=0 crossing
        ax.axvline(x_m0, linestyle="--", color="C0", alpha=0.8,
                   label=f"m=0 at {x_m0:.2f}")
        ax.axvspan(ci_m0[0], ci_m0[2], color="C0", alpha=0.08,
                   label="bootstrap 95% CI of m=0")

        if wilson_overlap_interval is not None:
            ax.axvspan(
                wilson_overlap_interval[0],
                wilson_overlap_interval[1],
                color="gold", alpha=0.18,
                label="Wilson-CI overlap area"
            )

        ax.set_xlabel(x_col)
        ax.set_ylabel("Order parameters")
        ax.set_title(
            f"Empirical Blume-Capel order parameters vs {x_col}\n"
            "Wilson errorbars (per-bin) + walk-level bootstrap bands"
        )
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="best")
        plt.tight_layout()
        plt.show()

        # --------------------------------------------------------
        # Plot 2: (m, q) trajectory
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(7, 7))

        mm = np.linspace(-1, 1, 200)
        ax.plot(mm, np.abs(mm), color="gray", linestyle="--", alpha=0.6,
                label="q = |m|  (no coexistence)")
        ax.axhline(1, color="gray", linestyle=":", alpha=0.4)
        ax.fill_between(mm, np.abs(mm), 1, color="gray", alpha=0.05)

        sc = ax.scatter(
            bin_df["m"], bin_df["q"],
            c=bin_df["x_mean"], cmap="coolwarm",
            s=bin_df["n"] * 2,
            edgecolor="black", linewidth=0.6,
        )
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label(x_col)

        # Connect bins in order of x
        ax.plot(bin_df["m"], bin_df["q"], color="gray", alpha=0.4, lw=1)

        # Mark m=0 and Var(S)/q peak
        ax.scatter(
            [bin_df["m"].iloc[i_chi_peak]], [bin_df["q"].iloc[i_chi_peak]],
            s=200, marker="*", color="purple", edgecolor="black",
            zorder=5, label=f"Var(S)/q peak ({x_chi_peak:.2f})"
        )

        ax.set_xlabel("m = P(U) - P(C)")
        ax.set_ylabel("q = P(U) + P(C)")
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(
            f"Empirical trajectory in (m, q) plane vs {x_col}\n"
            "Bubble size = bin count"
        )
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="upper center")
        plt.tight_layout()
        plt.show()

        # --------------------------------------------------------
        # Plot 3: Delta(T)
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(10, 4.8))

        ax.fill_between(
            x, bin_df["delta_lo_boot"], bin_df["delta_hi_boot"],
            color="C2", alpha=0.20, label="bootstrap 95% CI"
        )
        ax.plot(x, bin_df["delta"], "o-", color="C2", lw=2,
                label=r"$\Delta = q - |m|$")

        ax.axvline(x_delta_peak, linestyle="--", color="C2", alpha=0.8,
                   label=f"Delta peak at {x_delta_peak:.2f}")
        ax.axvspan(ci_dpk[0], ci_dpk[2], color="C2", alpha=0.08)

        ax.axvline(x_m0, linestyle=":", color="C0", alpha=0.6,
                   label=f"m=0 at {x_m0:.2f}")

        ax.set_xlabel(x_col)
        ax.set_ylabel(r"$\Delta(T)$")
        ax.set_title(f"Coexistence indicator from empirical bins")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

        # --------------------------------------------------------
        # Plot 4: Var(S) and Var(S)/q together
        # --------------------------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

        # Left: Var(S)
        ax = axes[0]
        ax.fill_between(
            x, bin_df["var_lo_boot"], bin_df["var_hi_boot"],
            color="purple", alpha=0.18, label="bootstrap 95% CI"
        )
        ax.plot(x, bin_df["var_S"], "o-", color="purple", lw=2,
                label=r"Var(S) = q - m$^2$")
        ax.axvline(x_var_peak, linestyle="--", color="purple", alpha=0.8,
                   label=f"Peak at {x_var_peak:.2f}")
        ax.axvspan(ci_vpk[0], ci_vpk[2], color="purple", alpha=0.08)
        ax.axvline(x_m0, linestyle=":", color="C0", alpha=0.6,
                   label=f"m=0 at {x_m0:.2f}")
        ax.set_xlabel(x_col)
        ax.set_ylabel("Var(S)")
        ax.set_title("Raw spin variance")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

        # Right: Var(S)/q
        ax = axes[1]
        ax.fill_between(
            x, bin_df["chi_lo_boot"], bin_df["chi_hi_boot"],
            color="darkorchid", alpha=0.18, label="bootstrap 95% CI"
        )
        ax.plot(x, bin_df["chi_tilde"], "o-", color="darkorchid", lw=2,
                label=r"$\tilde{\chi}$ = Var(S)/q")
        ax.axvline(x_chi_peak, linestyle="--", color="darkorchid", alpha=0.8,
                   label=f"Peak at {x_chi_peak:.2f}")
        ax.axvspan(ci_cpk[0], ci_cpk[2], color="darkorchid", alpha=0.08)
        ax.axvline(x_m0, linestyle=":", color="C0", alpha=0.6,
                   label=f"m=0 at {x_m0:.2f}")
        ax.set_xlabel(x_col)
        ax.set_ylabel(r"$\tilde{\chi}$")
        ax.set_title("Normalised susceptibility")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()

    # ============================================================
    # RETURN
    # ============================================================
    result = {
        "predictor": x_col,
        "bin_df": bin_df,
        "boot_m": boot_m,
        "boot_q": boot_q,
        "boot_delta": boot_delta,
        "boot_var": boot_var,
        "boot_chi": boot_chi,
        "x_m0": x_m0, "ci_m0": ci_m0,
        "wilson_overlap_interval": wilson_overlap_interval,
        "x_delta_peak": x_delta_peak, "ci_dpk": ci_dpk,
        "x_var_peak": x_var_peak, "ci_vpk": ci_vpk,
        "x_chi_peak": x_chi_peak, "ci_cpk": ci_cpk,
    }

    return result


# ============================================================
# RUN
# ============================================================

res_T = empirical_blume_capel(df, x_col=temp_corr_col)
# res_HDX = empirical_blume_capel(df, x_col=hdx_col)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# HELPERS
# ============================================================

def wilson_ci(k, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan

    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    phat = k / n

    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (
        z
        * np.sqrt((phat * (1 - phat) / n) + (z**2 / (4 * n**2)))
        / denom
    )

    return center - half, center + half


def linear_interp_crossing(x, y, target=0.0):
    """Linear interpolation of crossing point y(x) = target between bins."""
    y = np.asarray(y) - target
    x = np.asarray(x)

    valid = np.isfinite(y)
    if valid.sum() < 2:
        return np.nan, None

    x = x[valid]
    y = y[valid]

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))], None

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0, (x0, x1)

    crossing = x0 - y0 * (x1 - x0) / (y1 - y0)

    return crossing, (min(x0, x1), max(x0, x1))


def compute_order_params_from_counts(kC, kU, n):
    """Given counts of comfortable, uncomfortable and total in a bin,
    return m, q, delta, var_S, chi_tilde."""
    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    pC = kC / n
    pU = kU / n

    m = pU - pC
    q = pU + pC
    delta = q - abs(m)
    var_S = q - m**2
    chi_tilde = var_S / q if q > 1e-6 else np.nan

    return m, q, delta, var_S, chi_tilde


# ============================================================
# MAIN FUNCTION: EMPIRICAL BLUME-CAPEL WITH STRATIFIED BOOTSTRAP
# ============================================================

def empirical_blume_capel(
    df_in,
    x_col,
    y_col="comfort_3",
    cluster_col="walk",
    class_order=None,
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
    n_boot=500,
    chi_threshold=0.95,
    random_state=123,
    plot=True,
    run_jackknife=True,
):
    """
    Empirical Blume-Capel order parameters with two-source uncertainty:
      - Wilson CIs (within-bin binomial fluctuation)
      - Stratified bootstrap: resample VOTES WITHIN walks with replacement
        (preserves walk structure, measures conditional within-walk variability)
      - Optional jackknife: leave-one-walk-out robustness check

    The stratified bootstrap is more appropriate than naive walk-level
    bootstrap when the 48 walks are not a sample from a superpopulation
    but a fixed collection whose internal noise we want to characterise.
    """

    if class_order is None:
        class_order = ["comfortable_side", "neutral", "uncomfortable_side"]

    rng = np.random.default_rng(random_state)

    sub = df_in[[x_col, y_col, cluster_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty:
        print(f"No data for {x_col}")
        return None

    print(f"\nPredictor: {x_col}")
    print(f"N votes = {len(sub)}")
    print(f"N walks = {sub[cluster_col].nunique()}")
    print(sub[y_col].value_counts().reindex(class_order))

    # ----------------------------------------------------
    # Bin assignment on full sample
    # ----------------------------------------------------
    if binning == "quantile":
        sub["x_bin"] = pd.qcut(sub[x_col], q=n_bins, duplicates="drop")
    elif binning == "equal_width":
        sub["x_bin"] = pd.cut(sub[x_col], bins=n_bins, include_lowest=True)
    else:
        raise ValueError("binning must be 'quantile' or 'equal_width'")

    # ----------------------------------------------------
    # ORIGINAL ESTIMATE: empirical bins with Wilson CI
    # ----------------------------------------------------
    rows = []

    for b, g in sub.groupby("x_bin", observed=False):
        n = len(g)

        if n < min_bin_n:
            continue

        kC = (g[y_col] == "comfortable_side").sum()
        kN = (g[y_col] == "neutral").sum()
        kU = (g[y_col] == "uncomfortable_side").sum()

        pC = kC / n
        pN = kN / n
        pU = kU / n

        loC, hiC = wilson_ci(kC, n)
        loN, hiN = wilson_ci(kN, n)
        loU, hiU = wilson_ci(kU, n)

        m, q, delta, var_S, chi_tilde = compute_order_params_from_counts(
            kC, kU, n
        )

        m_lo = loU - hiC
        m_hi = hiU - loC
        q_lo = max(0.0, loU + loC)
        q_hi = min(1.0, hiU + hiC)

        rows.append({
            "bin": b,
            "bin_left":  b.left  if hasattr(b, "left")  else np.nan,
            "bin_right": b.right if hasattr(b, "right") else np.nan,
            "x_mean": g[x_col].mean(),
            "n": n,
            "pC": pC, "pN": pN, "pU": pU,
            "pC_lo": loC, "pC_hi": hiC,
            "pN_lo": loN, "pN_hi": hiN,
            "pU_lo": loU, "pU_hi": hiU,
            "m": m, "m_lo_wilson": m_lo, "m_hi_wilson": m_hi,
            "q": q, "q_lo_wilson": q_lo, "q_hi_wilson": q_hi,
            "delta": delta,
            "var_S": var_S,
            "chi_tilde": chi_tilde,
        })

    bin_df = pd.DataFrame(rows).sort_values("x_mean").reset_index(drop=True)

    if bin_df.empty:
        print("No valid bins after min_bin_n filtering.")
        return None

    bin_index = bin_df["bin"].tolist()
    n_bins_kept = len(bin_index)
    x_centers = bin_df["x_mean"].values

    # ----------------------------------------------------
    # STRATIFIED BOOTSTRAP
    # Resample votes WITHIN each walk with replacement.
    # This preserves the set of walks (no walk is lost or repeated)
    # and measures the conditional variability within each walk.
    # ----------------------------------------------------
    print(f"\nRunning stratified bootstrap (within-walk resampling)...")

    # Pre-index: list of indices per walk for fast resampling
    walk_indices = {
        w: np.array(idx)
        for w, idx in sub.groupby(cluster_col).indices.items()
    }
    walks = list(walk_indices.keys())

    sub_arr_y = sub[y_col].values
    sub_arr_bin = sub["x_bin"].values

    boot_m = np.full((n_boot, n_bins_kept), np.nan)
    boot_q = np.full((n_boot, n_bins_kept), np.nan)
    boot_delta = np.full((n_boot, n_bins_kept), np.nan)
    boot_var = np.full((n_boot, n_bins_kept), np.nan)
    boot_chi = np.full((n_boot, n_bins_kept), np.nan)

    boot_x_m0 = np.full(n_boot, np.nan)
    boot_x_delta_peak = np.full(n_boot, np.nan)
    boot_x_var_peak = np.full(n_boot, np.nan)
    boot_x_chi_peak = np.full(n_boot, np.nan)
    boot_window_lo = np.full(n_boot, np.nan)
    boot_window_hi = np.full(n_boot, np.nan)
    boot_window_width = np.full(n_boot, np.nan)

    for b_iter in range(n_boot):
        # Resample within each walk independently
        resampled_idx = np.concatenate([
            rng.choice(walk_indices[w], size=len(walk_indices[w]), replace=True)
            for w in walks
        ])

        y_b = sub_arr_y[resampled_idx]
        bin_b = sub_arr_bin[resampled_idx]

        # Compute per-bin order parameters
        for j, b_label in enumerate(bin_index):
            mask = bin_b == b_label
            n_g = mask.sum()

            if n_g < 1:
                continue

            y_g = y_b[mask]
            kC = (y_g == "comfortable_side").sum()
            kU = (y_g == "uncomfortable_side").sum()

            m_b, q_b, d_b, v_b, c_b = compute_order_params_from_counts(
                kC, kU, n_g
            )

            boot_m[b_iter, j] = m_b
            boot_q[b_iter, j] = q_b
            boot_delta[b_iter, j] = d_b
            boot_var[b_iter, j] = v_b
            boot_chi[b_iter, j] = c_b

        # Critical points per replica
        if np.isfinite(boot_m[b_iter]).sum() >= 2:
            x_m0_b, _ = linear_interp_crossing(
                x_centers, boot_m[b_iter], target=0.0
            )
            boot_x_m0[b_iter] = x_m0_b

        if np.isfinite(boot_delta[b_iter]).any():
            boot_x_delta_peak[b_iter] = x_centers[
                np.nanargmax(boot_delta[b_iter])
            ]
        if np.isfinite(boot_var[b_iter]).any():
            boot_x_var_peak[b_iter] = x_centers[
                np.nanargmax(boot_var[b_iter])
            ]
        if np.isfinite(boot_chi[b_iter]).any():
            boot_x_chi_peak[b_iter] = x_centers[
                np.nanargmax(boot_chi[b_iter])
            ]

        # Critical window per replica
        chi_vals = boot_chi[b_iter]
        above = np.isfinite(chi_vals) & (chi_vals >= chi_threshold)
        if above.any():
            idx_above = np.where(above)[0]
            boot_window_lo[b_iter] = x_centers[idx_above[0]]
            boot_window_hi[b_iter] = x_centers[idx_above[-1]]
            boot_window_width[b_iter] = (
                boot_window_hi[b_iter] - boot_window_lo[b_iter]
            )

    # ----------------------------------------------------
    # CIs per bin from stratified bootstrap
    # ----------------------------------------------------
    bin_df["m_lo_boot"]  = np.nanpercentile(boot_m, 2.5, axis=0)
    bin_df["m_hi_boot"]  = np.nanpercentile(boot_m, 97.5, axis=0)
    bin_df["q_lo_boot"]  = np.nanpercentile(boot_q, 2.5, axis=0)
    bin_df["q_hi_boot"]  = np.nanpercentile(boot_q, 97.5, axis=0)
    bin_df["delta_lo_boot"] = np.nanpercentile(boot_delta, 2.5, axis=0)
    bin_df["delta_hi_boot"] = np.nanpercentile(boot_delta, 97.5, axis=0)
    bin_df["var_lo_boot"]   = np.nanpercentile(boot_var, 2.5, axis=0)
    bin_df["var_hi_boot"]   = np.nanpercentile(boot_var, 97.5, axis=0)
    bin_df["chi_lo_boot"]   = np.nanpercentile(boot_chi, 2.5, axis=0)
    bin_df["chi_hi_boot"]   = np.nanpercentile(boot_chi, 97.5, axis=0)

    # ----------------------------------------------------
    # Critical points from ORIGINAL estimate
    # ----------------------------------------------------
    x_m0, m0_interval = linear_interp_crossing(
        x_centers, bin_df["m"].values, target=0.0
    )

    i_delta_peak = int(np.nanargmax(bin_df["delta"].values))
    x_delta_peak = x_centers[i_delta_peak]

    i_var_peak = int(np.nanargmax(bin_df["var_S"].values))
    x_var_peak = x_centers[i_var_peak]

    i_chi_peak = int(np.nanargmax(bin_df["chi_tilde"].values))
    x_chi_peak = x_centers[i_chi_peak]

    ci_m0  = np.nanpercentile(boot_x_m0,  [2.5, 50, 97.5])
    ci_dpk = np.nanpercentile(boot_x_delta_peak, [2.5, 50, 97.5])
    ci_vpk = np.nanpercentile(boot_x_var_peak,   [2.5, 50, 97.5])
    ci_cpk = np.nanpercentile(boot_x_chi_peak,   [2.5, 50, 97.5])

    # Critical window from ORIGINAL estimate
    chi_vals_orig = bin_df["chi_tilde"].values
    above_orig = chi_vals_orig >= chi_threshold

    if above_orig.any():
        idx_above_orig = np.where(above_orig)[0]
        crit_window_lo = x_centers[idx_above_orig[0]]
        crit_window_hi = x_centers[idx_above_orig[-1]]
        crit_window_width = crit_window_hi - crit_window_lo
    else:
        crit_window_lo = np.nan
        crit_window_hi = np.nan
        crit_window_width = np.nan

    # Bootstrap CIs for the critical window
    ci_window_lo = np.nanpercentile(boot_window_lo, [2.5, 50, 97.5])
    ci_window_hi = np.nanpercentile(boot_window_hi, [2.5, 50, 97.5])
    ci_window_width = np.nanpercentile(boot_window_width, [2.5, 50, 97.5])

    # Wilson-CI overlap area
    ci_overlap = (
        (bin_df["pC_lo"] <= bin_df["pU_hi"])
        &
        (bin_df["pU_lo"] <= bin_df["pC_hi"])
    )
    overlap_bins = bin_df[ci_overlap]
    if len(overlap_bins) > 0:
        wilson_overlap_interval = (
            overlap_bins["bin_left"].min(),
            overlap_bins["bin_right"].max(),
        )
    else:
        wilson_overlap_interval = None

    # ----------------------------------------------------
    # JACKKNIFE: leave-one-walk-out robustness check
    # ----------------------------------------------------
    jack_results = None

    if run_jackknife:
        print("Running leave-one-walk-out jackknife...")

        jack_x_m0 = []
        jack_x_chi_peak = []
        jack_window_lo = []
        jack_window_hi = []
        jack_window_width = []
        jack_walk_id = []

        for w in walks:
            sub_jk = sub[sub[cluster_col] != w]

            rows_jk = []
            for b_label in bin_index:
                g = sub_jk[sub_jk["x_bin"] == b_label]
                n_g = len(g)
                if n_g < 1:
                    rows_jk.append({
                        "x_mean": np.nan, "m": np.nan, "chi": np.nan,
                    })
                    continue
                kC = (g[y_col] == "comfortable_side").sum()
                kU = (g[y_col] == "uncomfortable_side").sum()
                m_j, q_j, _, v_j, c_j = compute_order_params_from_counts(
                    kC, kU, n_g
                )
                rows_jk.append({
                    "x_mean": g[x_col].mean(),
                    "m": m_j,
                    "chi": c_j,
                })

            jk_df = pd.DataFrame(rows_jk)
            x_jk = jk_df["x_mean"].values
            m_jk = jk_df["m"].values
            chi_jk = jk_df["chi"].values

            x_m0_jk, _ = linear_interp_crossing(x_jk, m_jk, target=0.0)
            jack_x_m0.append(x_m0_jk)

            if np.isfinite(chi_jk).any():
                jack_x_chi_peak.append(x_jk[np.nanargmax(chi_jk)])
                above_jk = np.isfinite(chi_jk) & (chi_jk >= chi_threshold)
                if above_jk.any():
                    idx_jk = np.where(above_jk)[0]
                    jack_window_lo.append(x_jk[idx_jk[0]])
                    jack_window_hi.append(x_jk[idx_jk[-1]])
                    jack_window_width.append(x_jk[idx_jk[-1]] - x_jk[idx_jk[0]])
                else:
                    jack_window_lo.append(np.nan)
                    jack_window_hi.append(np.nan)
                    jack_window_width.append(np.nan)
            else:
                jack_x_chi_peak.append(np.nan)
                jack_window_lo.append(np.nan)
                jack_window_hi.append(np.nan)
                jack_window_width.append(np.nan)

            jack_walk_id.append(w)

        jack_results = pd.DataFrame({
            "walk_removed": jack_walk_id,
            "x_m0": jack_x_m0,
            "x_chi_peak": jack_x_chi_peak,
            "window_lo": jack_window_lo,
            "window_hi": jack_window_hi,
            "window_width": jack_window_width,
        })

    # ----------------------------------------------------
    # Print summary
    # ----------------------------------------------------
    print("\n" + "=" * 60)
    print("EMPIRICAL BLUME-CAPEL RESULTS")
    print("=" * 60)
    print("\nCritical points (stratified within-walk bootstrap 95% CI):")
    print(
        f"  m = 0:           {x_m0:.3f}   "
        f"[{ci_m0[0]:.3f}, {ci_m0[2]:.3f}]"
    )
    if wilson_overlap_interval is not None:
        print(
            f"  Wilson-CI overlap area: "
            f"[{wilson_overlap_interval[0]:.3f}, "
            f"{wilson_overlap_interval[1]:.3f}]"
        )
    print(
        f"  Delta peak:      {x_delta_peak:.3f}   "
        f"[{ci_dpk[0]:.3f}, {ci_dpk[2]:.3f}]"
    )
    print(
        f"  Var(S) peak:     {x_var_peak:.3f}   "
        f"[{ci_vpk[0]:.3f}, {ci_vpk[2]:.3f}]"
    )
    print(
        f"  Var(S)/q peak:   {x_chi_peak:.3f}   "
        f"[{ci_cpk[0]:.3f}, {ci_cpk[2]:.3f}]"
    )

    print(f"\nCritical window (chi_tilde >= {chi_threshold}):")
    print(
        f"  Lower edge: {crit_window_lo:.3f}   "
        f"[{ci_window_lo[0]:.3f}, {ci_window_lo[2]:.3f}]"
    )
    print(
        f"  Upper edge: {crit_window_hi:.3f}   "
        f"[{ci_window_hi[0]:.3f}, {ci_window_hi[2]:.3f}]"
    )
    print(
        f"  Width:      {crit_window_width:.3f}   "
        f"[{ci_window_width[0]:.3f}, {ci_window_width[2]:.3f}]"
    )

    if jack_results is not None:
        print("\nJackknife robustness (leave-one-walk-out):")
        jk_m0 = jack_results["x_m0"].dropna()
        jk_w  = jack_results["window_width"].dropna()
        print(
            f"  m=0 sensitivity:           "
            f"std={jk_m0.std():.3f}, "
            f"max deviation={(jk_m0 - x_m0).abs().max():.3f}"
        )
        if len(jk_w) > 0:
            print(
                f"  Window width sensitivity:  "
                f"std={jk_w.std():.3f}, "
                f"range=[{jk_w.min():.3f}, {jk_w.max():.3f}]"
            )
        # Most influential walks for m=0
        jack_results["m0_deviation"] = (jack_results["x_m0"] - x_m0).abs()
        influential = jack_results.nlargest(3, "m0_deviation")
        print("  Top 3 influential walks for m=0:")
        for _, row in influential.iterrows():
            print(
                f"    {row['walk_removed']}: removing shifts m=0 by "
                f"{row['x_m0'] - x_m0:+.3f}"
            )

      # ============================================================
    # PLOTS
    # ============================================================
    if plot:
        fig, ax = plt.subplots(figsize=(9, 5))

        x = bin_df["x_mean"].values

        # ----------------------------------------------------
        # m with bootstrap band only
        # ----------------------------------------------------
        ax.fill_between(
            x,
            bin_df["m_lo_boot"],
            bin_df["m_hi_boot"],
            color="C0",
            alpha=0.18,
            label=r"$m$ bootstrap 95% CI"
        )

        ax.plot(
            x,
            bin_df["m"],
            "o-",
            color="C0",
            lw=2,
            markersize=5,
            label=r"$m = P(U) - P(C)$"
        )

        # ----------------------------------------------------
        # q with bootstrap band only
        # ----------------------------------------------------
        ax.fill_between(
            x,
            bin_df["q_lo_boot"],
            bin_df["q_hi_boot"],
            color="C3",
            alpha=0.18,
            label=r"$q$ bootstrap 95% CI"
        )

        ax.plot(
            x,
            bin_df["q"],
            "s-",
            color="C3",
            lw=2,
            markersize=5,
            label=r"$q = P(U) + P(C)$"
        )

        # ----------------------------------------------------
        # Reference line
        # ----------------------------------------------------
        ax.axhline(
            0,
            linestyle="--",
            color="gray",
            alpha=0.6,
            lw=1
        )

        # ----------------------------------------------------
        # Bootstrap band of the m = 0 crossing only
        # ----------------------------------------------------
        ax.axvline(
            x_m0,
            linestyle="--",
            color="C0",
            alpha=0.8,
            lw=1.5,
            label=fr"$m=0$ at {x_m0:.2f}"
        )

        if np.all(np.isfinite(ci_m0[[0, 2]])):
            ax.axvspan(
                ci_m0[0],
                ci_m0[2],
                color="C0",
                alpha=0.08,
                label=fr"bootstrap 95% CI of $m=0$: "
                      fr"[{ci_m0[0]:.2f}, {ci_m0[2]:.2f}]"
            )

        ax.set_xlabel(x_col)
        ax.set_ylabel("Order parameters")

        ax.set_title(
            fr"Empirical Blume-Capel order parameters vs {x_col}"
            "\nBootstrap uncertainty only"
        )

        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, loc="best")

        plt.tight_layout()
        plt.show()

    # ============================================================
    # RETURN
    # ============================================================
    result = {
        "predictor": x_col,
        "bin_df": bin_df,
        "boot_m": boot_m,
        "boot_q": boot_q,
        "boot_delta": boot_delta,
        "boot_var": boot_var,
        "boot_chi": boot_chi,
        "x_m0": x_m0, "ci_m0": ci_m0,
        "wilson_overlap_interval": wilson_overlap_interval,
        "x_delta_peak": x_delta_peak, "ci_dpk": ci_dpk,
        "x_var_peak": x_var_peak, "ci_vpk": ci_vpk,
        "x_chi_peak": x_chi_peak, "ci_cpk": ci_cpk,
        "crit_window_lo": crit_window_lo,
        "crit_window_hi": crit_window_hi,
        "crit_window_width": crit_window_width,
        "ci_window_lo": ci_window_lo,
        "ci_window_hi": ci_window_hi,
        "ci_window_width": ci_window_width,
        "jackknife": jack_results,
    }

    return result


# ============================================================
# RUN
# ============================================================
res_T = empirical_blume_capel(
    df,
    x_col=temp_corr_col,
    run_jackknife=False
)

res_HDX = empirical_blume_capel(
    df,
    x_col=hdx_col,
    run_jackknife=False
)

In [ ]:
def compute_bc_fields_from_counts(kC, kN, kU, eps=0.5):
    """
    Effective single-site Blume-Capel fields from empirical probabilities.

    P(C) = P(S=-1)
    P(N) = P(S=0)
    P(U) = P(S=+1)

    beta h = 1/2 ln[P(U)/P(C)]
    beta D = -1/2 ln[P(U)P(C)/P(N)^2]

    eps=0.5 is Jeffreys smoothing to avoid log(0).
    """
    total = kC + kN + kU

    if total == 0:
        return np.nan, np.nan

    pC = (kC + eps) / (total + 3 * eps)
    pN = (kN + eps) / (total + 3 * eps)
    pU = (kU + eps) / (total + 3 * eps)

    beta_h = 0.5 * np.log(pU / pC)
    beta_D = -0.5 * np.log((pU * pC) / (pN**2))

    return beta_h, beta_D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# HELPERS
# ============================================================

def linear_interp_crossing(x, y, target=0.0):
    """
    Linear interpolation of crossing point y(x) = target between bins.
    If there is no true sign change, returns the x closest to target.
    """
    y = np.asarray(y, dtype=float) - target
    x = np.asarray(x, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan, None

    x = x[valid]
    y = y[valid]

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))], None

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0, (x0, x1)

    crossing = x0 - y0 * (x1 - x0) / (y1 - y0)

    return crossing, (min(x0, x1), max(x0, x1))


def compute_order_params_from_counts(kC, kU, n):
    """
    Given counts of comfortable, uncomfortable and total in a bin,
    return m, q, delta, var_S, chi_tilde.

    C = comfortable side
    N = neutral
    U = uncomfortable side

    m = P(U) - P(C)
    q = P(U) + P(C)
    """
    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    pC = kC / n
    pU = kU / n

    m = pU - pC
    q = pU + pC
    delta = q - abs(m)
    var_S = q - m**2
    chi_tilde = var_S / q if q > 1e-6 else np.nan

    return m, q, delta, var_S, chi_tilde


def compute_bc_fields_from_counts(kC, kN, kU, smoothing=0.5):
    """
    Effective single-site Blume-Capel fields from empirical probabilities.

    P(C) = P(S = -1)
    P(N) = P(S = 0)
    P(U) = P(S = +1)

    beta h =  1/2 ln[P(U) / P(C)]
    beta D = -1/2 ln[P(U)P(C) / P(N)^2]

    smoothing=0.5 is Jeffreys smoothing.
    It avoids log(0) when a bootstrap/bin has zero counts.
    """
    total = kC + kN + kU

    if total == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    pC = (kC + smoothing) / (total + 3 * smoothing)
    pN = (kN + smoothing) / (total + 3 * smoothing)
    pU = (kU + smoothing) / (total + 3 * smoothing)

    beta_h = 0.5 * np.log(pU / pC)
    beta_D = -0.5 * np.log((pU * pC) / (pN**2))

    return beta_h, beta_D, pC, pN, pU


def get_transition_band_from_bootstrap(
    x,
    m_lo,
    m_hi,
    mode="points"
):
    """
    Transition band from the bootstrap envelope of m.

    mode="points":
        The band goes from the first to the last bin centre whose
        bootstrap CI contains zero.

        This is the most literal version of:
        "incloure tots els punts que tallen l'eix 0".

    mode="interpolate":
        Similar, but the edges are linearly interpolated using the
        bootstrap envelope. This gives a smoother but sometimes wider band.
    """
    x = np.asarray(x, dtype=float)
    m_lo = np.asarray(m_lo, dtype=float)
    m_hi = np.asarray(m_hi, dtype=float)

    valid = np.isfinite(x) & np.isfinite(m_lo) & np.isfinite(m_hi)

    if valid.sum() < 1:
        return np.nan, np.nan, np.zeros_like(x, dtype=bool)

    zero_mask = valid & (m_lo <= 0) & (m_hi >= 0)

    if not zero_mask.any():
        return np.nan, np.nan, zero_mask

    idx = np.where(zero_mask)[0]
    first = idx[0]
    last = idx[-1]

    if mode == "points":
        return x[first], x[last], zero_mask

    elif mode == "interpolate":
        # Left edge: where upper bootstrap envelope crosses 0
        if first > 0 and np.isfinite(m_hi[first - 1]):
            x0, x1 = x[first - 1], x[first]
            y0, y1 = m_hi[first - 1], m_hi[first]

            if y1 != y0:
                lo = x0 + (0 - y0) * (x1 - x0) / (y1 - y0)
            else:
                lo = x[first]
        else:
            lo = x[first]

        # Right edge: where lower bootstrap envelope crosses 0
        if last < len(x) - 1 and np.isfinite(m_lo[last + 1]):
            x0, x1 = x[last], x[last + 1]
            y0, y1 = m_lo[last], m_lo[last + 1]

            if y1 != y0:
                hi = x0 + (0 - y0) * (x1 - x0) / (y1 - y0)
            else:
                hi = x[last]
        else:
            hi = x[last]

        return lo, hi, zero_mask

    else:
        raise ValueError("mode must be 'points' or 'interpolate'")


# ============================================================
# MAIN FUNCTION
# ============================================================

def empirical_blume_capel(
    df_in,
    x_col,
    y_col="comfort_3",
    cluster_col="walk",
    class_order=None,
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
    n_boot=500,
    random_state=123,
    smoothing=0.5,
    transition_band_mode="points",
    plot=True,
):
    """
    Empirical Blume-Capel order parameters with stratified bootstrap.

    Outputs:
      - m = P(U) - P(C)
      - q = P(U) + P(C)
      - beta h = 1/2 ln[P(U) / P(C)]
      - beta D = -1/2 ln[P(U)P(C) / P(N)^2]

    Bootstrap:
      - Resamples votes within each walk.
      - Keeps the set of walks fixed.
      - Measures within-walk conditional uncertainty.

    Plot:
      - Plot 1: m and q with bootstrap bands only.
      - Plot 2: beta h and beta D with bootstrap bands only.
      - No Wilson errorbars.
      - Transition band = bins whose m bootstrap CI contains zero.
    """

    if class_order is None:
        class_order = ["comfortable_side", "neutral", "uncomfortable_side"]

    rng = np.random.default_rng(random_state)

    sub = df_in[[x_col, y_col, cluster_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty:
        print(f"No data for {x_col}")
        return None

    print("\n" + "=" * 70)
    print(f"Predictor: {x_col}")
    print("=" * 70)
    print(f"N votes = {len(sub)}")
    print(f"N walks = {sub[cluster_col].nunique()}")
    print(sub[y_col].value_counts().reindex(class_order))

    # ----------------------------------------------------
    # Bin assignment
    # ----------------------------------------------------
    if binning == "quantile":
        sub["x_bin"] = pd.qcut(
            sub[x_col],
            q=n_bins,
            duplicates="drop"
        )
    elif binning == "equal_width":
        sub["x_bin"] = pd.cut(
            sub[x_col],
            bins=n_bins,
            include_lowest=True
        )
    else:
        raise ValueError("binning must be 'quantile' or 'equal_width'")

    # ----------------------------------------------------
    # Original empirical estimate
    # ----------------------------------------------------
    rows = []

    for b, g in sub.groupby("x_bin", observed=False):
        n = len(g)

        if n < min_bin_n:
            continue

        kC = (g[y_col] == "comfortable_side").sum()
        kN = (g[y_col] == "neutral").sum()
        kU = (g[y_col] == "uncomfortable_side").sum()

        pC_raw = kC / n
        pN_raw = kN / n
        pU_raw = kU / n

        m, q, delta, var_S, chi_tilde = compute_order_params_from_counts(
            kC,
            kU,
            n
        )

        beta_h, beta_D, pC_s, pN_s, pU_s = compute_bc_fields_from_counts(
            kC,
            kN,
            kU,
            smoothing=smoothing
        )

        rows.append({
            "bin": b,
            "bin_left": b.left if hasattr(b, "left") else np.nan,
            "bin_right": b.right if hasattr(b, "right") else np.nan,
            "x_mean": g[x_col].mean(),
            "x_median": g[x_col].median(),
            "n": n,

            "kC": kC,
            "kN": kN,
            "kU": kU,

            "pC": pC_raw,
            "pN": pN_raw,
            "pU": pU_raw,

            "pC_smooth": pC_s,
            "pN_smooth": pN_s,
            "pU_smooth": pU_s,

            "m": m,
            "q": q,
            "delta": delta,
            "var_S": var_S,
            "chi_tilde": chi_tilde,

            "beta_h": beta_h,
            "beta_D": beta_D,
        })

    bin_df = pd.DataFrame(rows).sort_values("x_mean").reset_index(drop=True)

    if bin_df.empty:
        print("No valid bins after min_bin_n filtering.")
        return None

    bin_index = bin_df["bin"].tolist()
    n_bins_kept = len(bin_index)
    x_centers = bin_df["x_mean"].values

    # ----------------------------------------------------
    # Stratified bootstrap: resample votes within walks
    # ----------------------------------------------------
    print("\nRunning stratified bootstrap...")

    walk_indices = {
        w: np.array(idx)
        for w, idx in sub.groupby(cluster_col).indices.items()
    }

    walks = list(walk_indices.keys())

    sub_arr_y = sub[y_col].values
    sub_arr_bin = sub["x_bin"].values

    boot_m = np.full((n_boot, n_bins_kept), np.nan)
    boot_q = np.full((n_boot, n_bins_kept), np.nan)
    boot_delta = np.full((n_boot, n_bins_kept), np.nan)
    boot_var = np.full((n_boot, n_bins_kept), np.nan)
    boot_chi = np.full((n_boot, n_bins_kept), np.nan)

    boot_beta_h = np.full((n_boot, n_bins_kept), np.nan)
    boot_beta_D = np.full((n_boot, n_bins_kept), np.nan)

    boot_x_m0 = np.full(n_boot, np.nan)

    for b_iter in range(n_boot):

        resampled_idx = np.concatenate([
            rng.choice(
                walk_indices[w],
                size=len(walk_indices[w]),
                replace=True
            )
            for w in walks
        ])

        y_b = sub_arr_y[resampled_idx]
        bin_b = sub_arr_bin[resampled_idx]

        for j, b_label in enumerate(bin_index):
            mask = bin_b == b_label
            n_g = mask.sum()

            if n_g < 1:
                continue

            y_g = y_b[mask]

            kC = (y_g == "comfortable_side").sum()
            kN = (y_g == "neutral").sum()
            kU = (y_g == "uncomfortable_side").sum()

            m_b, q_b, d_b, v_b, c_b = compute_order_params_from_counts(
                kC,
                kU,
                n_g
            )

            bh_b, bD_b, _, _, _ = compute_bc_fields_from_counts(
                kC,
                kN,
                kU,
                smoothing=smoothing
            )

            boot_m[b_iter, j] = m_b
            boot_q[b_iter, j] = q_b
            boot_delta[b_iter, j] = d_b
            boot_var[b_iter, j] = v_b
            boot_chi[b_iter, j] = c_b

            boot_beta_h[b_iter, j] = bh_b
            boot_beta_D[b_iter, j] = bD_b

        if np.isfinite(boot_m[b_iter]).sum() >= 2:
            x_m0_b, _ = linear_interp_crossing(
                x_centers,
                boot_m[b_iter],
                target=0.0
            )
            boot_x_m0[b_iter] = x_m0_b

    # ----------------------------------------------------
    # Bootstrap CIs per bin
    # ----------------------------------------------------
    bin_df["m_lo_boot"] = np.nanpercentile(boot_m, 2.5, axis=0)
    bin_df["m_hi_boot"] = np.nanpercentile(boot_m, 97.5, axis=0)

    bin_df["q_lo_boot"] = np.nanpercentile(boot_q, 2.5, axis=0)
    bin_df["q_hi_boot"] = np.nanpercentile(boot_q, 97.5, axis=0)

    bin_df["delta_lo_boot"] = np.nanpercentile(boot_delta, 2.5, axis=0)
    bin_df["delta_hi_boot"] = np.nanpercentile(boot_delta, 97.5, axis=0)

    bin_df["var_lo_boot"] = np.nanpercentile(boot_var, 2.5, axis=0)
    bin_df["var_hi_boot"] = np.nanpercentile(boot_var, 97.5, axis=0)

    bin_df["chi_lo_boot"] = np.nanpercentile(boot_chi, 2.5, axis=0)
    bin_df["chi_hi_boot"] = np.nanpercentile(boot_chi, 97.5, axis=0)

    bin_df["beta_h_lo_boot"] = np.nanpercentile(boot_beta_h, 2.5, axis=0)
    bin_df["beta_h_hi_boot"] = np.nanpercentile(boot_beta_h, 97.5, axis=0)

    bin_df["beta_D_lo_boot"] = np.nanpercentile(boot_beta_D, 2.5, axis=0)
    bin_df["beta_D_hi_boot"] = np.nanpercentile(boot_beta_D, 97.5, axis=0)

    # ----------------------------------------------------
    # Critical crossing and transition band
    # ----------------------------------------------------
    x_m0, m0_interval = linear_interp_crossing(
        x_centers,
        bin_df["m"].values,
        target=0.0
    )

    ci_m0 = np.nanpercentile(
        boot_x_m0,
        [2.5, 50, 97.5]
    )

    transition_lo, transition_hi, transition_mask = get_transition_band_from_bootstrap(
        x_centers,
        bin_df["m_lo_boot"].values,
        bin_df["m_hi_boot"].values,
        mode=transition_band_mode
    )

    bin_df["m_boot_crosses_zero"] = transition_mask

    # ----------------------------------------------------
    # Summary
    # ----------------------------------------------------
    print("\nSummary:")
    print(f"  m = 0 crossing: {x_m0:.3f}")
    print(
        f"  bootstrap distribution of m=0 crossings: "
        f"[{ci_m0[0]:.3f}, {ci_m0[2]:.3f}]"
    )

    if np.isfinite(transition_lo) and np.isfinite(transition_hi):
        print(
            f"  transition band from bootstrap envelope: "
            f"[{transition_lo:.3f}, {transition_hi:.3f}]"
        )
        print(
            f"  bins crossing zero: "
            f"{int(bin_df['m_boot_crosses_zero'].sum())} / {len(bin_df)}"
        )
    else:
        print("  no bootstrap bin contains m = 0")

    # ============================================================
    # PLOTS
    # ============================================================
    if plot:
        # -----------------------------
        # Custom palette
        # -----------------------------
        COL_RED = "#F52837"      # Strawberry Red
        COL_AQUA = "#9CE5E8"     # Icy Aqua
        COL_BLUE = "#2C7AB5"     # Bright Teal Blue
        COL_INDIGO = "#152F61"   # Twilight Indigo

        x = bin_df["x_mean"].values

        # ====================================================
        # PLOT 1: m and q
        # ====================================================
        fig, ax = plt.subplots(figsize=(9, 5))

        ax.fill_between(
            x,
            bin_df["m_lo_boot"],
            bin_df["m_hi_boot"],
            color=COL_BLUE,
            alpha=0.18,
            label=r"$m$ bootstrap 95% CI"
        )

        ax.plot(
            x,
            bin_df["m"],
            "o-",
            color=COL_INDIGO,
            lw=2,
            markersize=5,
            label=r"$m = P(U) - P(C)$"
        )

        ax.fill_between(
            x,
            bin_df["q_lo_boot"],
            bin_df["q_hi_boot"],
            color=COL_RED,
            alpha=0.12,
            label=r"$q$ bootstrap 95% CI"
        )

        ax.plot(
            x,
            bin_df["q"],
            "s-",
            color=COL_RED,
            lw=2,
            markersize=5,
            label=r"$q = P(U) + P(C)$"
        )

        ax.axhline(
            0,
            linestyle="--",
            color="gray",
            alpha=0.6,
            lw=1
        )

        ax.axvline(
            x_m0,
            linestyle="--",
            color=COL_BLUE,
            alpha=0.9,
            lw=1.5,
            label=fr"$m=0$ at {x_m0:.2f}"
        )

        if np.isfinite(transition_lo) and np.isfinite(transition_hi):
            ax.axvspan(
                transition_lo,
                transition_hi,
                color=COL_AQUA,
                alpha=0.28,
                label=fr"transition band: "
                      fr"[{transition_lo:.2f}, {transition_hi:.2f}]"
            )

        ax.set_xlabel(x_col, fontsize=12, color=COL_INDIGO)
        ax.set_ylabel("Order parameters", fontsize=12, color=COL_INDIGO)

        ax.set_title(
            fr"Empirical Blume-Capel order parameters vs {x_col}"
            "\nBootstrap uncertainty only",
            fontsize=15,
            color=COL_INDIGO
        )

        ax.tick_params(axis="both", colors=COL_INDIGO)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, loc="best", frameon=True)

        plt.tight_layout()
        plt.show()

        # ====================================================
        # PLOT 2: beta h and beta D
        # ====================================================
        fig, ax = plt.subplots(figsize=(9, 5))

        ax.fill_between(
            x,
            bin_df["beta_h_lo_boot"],
            bin_df["beta_h_hi_boot"],
            color=COL_BLUE,
            alpha=0.18,
            label=r"$\beta h$ bootstrap 95% CI"
        )

        ax.plot(
            x,
            bin_df["beta_h"],
            "o-",
            color=COL_INDIGO,
            lw=2,
            markersize=5,
            label=r"$\beta h = \frac{1}{2}\ln\frac{P(U)}{P(C)}$"
        )

        ax.fill_between(
            x,
            bin_df["beta_D_lo_boot"],
            bin_df["beta_D_hi_boot"],
            color=COL_RED,
            alpha=0.12,
            label=r"$\beta D$ bootstrap 95% CI"
        )

        ax.plot(
            x,
            bin_df["beta_D"],
            "s-",
            color=COL_RED,
            lw=2,
            markersize=5,
            label=r"$\beta D = -\frac{1}{2}\ln\frac{P(U)P(C)}{P(N)^2}$"
        )

        ax.axhline(
            0,
            linestyle="--",
            color="gray",
            alpha=0.6,
            lw=1
        )

        ax.axvline(
            x_m0,
            linestyle="--",
            color=COL_BLUE,
            alpha=0.9,
            lw=1.5,
            label=fr"$m=0$ at {x_m0:.2f}"
        )

        if np.isfinite(transition_lo) and np.isfinite(transition_hi):
            ax.axvspan(
                transition_lo,
                transition_hi,
                color=COL_AQUA,
                alpha=0.28,
                label=fr"transition band: "
                      fr"[{transition_lo:.2f}, {transition_hi:.2f}]"
            )

        ax.set_xlabel(x_col, fontsize=12, color=COL_INDIGO)
        ax.set_ylabel(r"Effective fields", fontsize=12, color=COL_INDIGO)

        ax.set_title(
            fr"Effective Blume-Capel fields vs {x_col}"
            "\nFields inferred from empirical probabilities",
            fontsize=15,
            color=COL_INDIGO
        )

        ax.tick_params(axis="both", colors=COL_INDIGO)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, loc="best", frameon=True)

        plt.tight_layout()
        plt.show()

    # ============================================================
    # RETURN
    # ============================================================
    result = {
        "predictor": x_col,
        "bin_df": bin_df,

        "boot_m": boot_m,
        "boot_q": boot_q,
        "boot_delta": boot_delta,
        "boot_var": boot_var,
        "boot_chi": boot_chi,
        "boot_beta_h": boot_beta_h,
        "boot_beta_D": boot_beta_D,

        "x_m0": x_m0,
        "ci_m0": ci_m0,

        "transition_lo": transition_lo,
        "transition_hi": transition_hi,
        "transition_band_mode": transition_band_mode,

        "smoothing": smoothing,
    }

    return result


# ============================================================
# RUN
# ============================================================

res_T = empirical_blume_capel(
    df,
    x_col=temp_corr_col,
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
    n_boot=500,
    smoothing=0.5,
    transition_band_mode="points",
    plot=True,
)

res_HDX = empirical_blume_capel(
    df,
    x_col=hdx_col,
    n_bins=12,
    min_bin_n=20,
    binning="quantile",
    n_boot=500,
    smoothing=0.5,
    transition_band_mode="points",
    plot=True,
)

In [ ]:
# ====================================================
# PLOT 1: m and q — two stacked subplots
# Enganxa aquesta cel·la just després d'haver corregut empirical_blume_capel
# res_T és el resultat retornat per la funció
# ====================================================

import numpy as np
import matplotlib.pyplot as plt

# ── Agafa les variables del resultat ─────────────────────────────────────────
bin_df        = res_T["bin_df"]
x             = bin_df["x_mean"].values
x_m0          = res_T["x_m0"]
transition_lo = res_T["transition_lo"]
transition_hi = res_T["transition_hi"]

# ── Paleta ───────────────────────────────────────────────────────────────────
COL_AQUA      = "#9CE5E8"   # Icy Aqua        — banda de transició
COL_BLUE      = "#2C7AB5"   # Bright Teal Blue — línia m
COL_DARK      = "#152F61"   # Twilight Indigo  — eixos / etiquetes
COL_ORANGE    = "#E07B39"   # Burnt Orange     — línia q (colorblind-safe)
COL_BLUE_FILL = "#C5DCF0"   # Blau pàl·lid     — CI de m
COL_ORG_FILL  = "#F5D0B5"   # Taronja pàl·lid  — CI de q

# ── Figura ───────────────────────────────────────────────────────────────────
# ── Calcula els rangs abans de crear la figura ────────────────────────────────
m_range = (
    min(bin_df["m_lo_boot"].min(), -0.05),
    max(bin_df["m_hi_boot"].max(),  0.05)
)
q_range = (
    max(0.0, bin_df["q_lo_boot"].min() - 0.04),
    min(1.0, bin_df["q_hi_boot"].max() + 0.04)
)

m_span = m_range[1] - m_range[0]
q_span = q_range[1] - q_range[0]

# height_ratios proporcionals als rangs → mateixa escala visual per unitat
fig, (ax_m, ax_q) = plt.subplots(
    2, 1,
    figsize=(7, 3 + 7 * q_span / m_span),   # alçada total proporcional
    sharex=True,
    gridspec_kw={
        "hspace": 0.08,
        "height_ratios": [m_span, q_span]    # ← clau
    }
)

# ── Fixa els límits explícitament ─────────────────────────────────────────────
ax_m.set_ylim(*m_range)
ax_q.set_ylim(*q_range)

# ... (resta del codi igual que abans) ...
# ── Elements comuns ───────────────────────────────────────────────────────────
for ax in (ax_m, ax_q):
    if np.isfinite(transition_lo) and np.isfinite(transition_hi):
        ax.axvspan(transition_lo, transition_hi,
                   color=COL_AQUA, alpha=0.40, zorder=0)
    ax.axvline(x_m0, linestyle=":", color=COL_DARK,
               alpha=0.50, lw=1.2, zorder=1)
    ax.grid(True, alpha=0.18, zorder=0)
    ax.tick_params(axis="both", colors=COL_DARK, labelsize=10)
    for spine in ax.spines.values():
        spine.set_edgecolor(COL_DARK)
        spine.set_linewidth(0.8)

# ── Panell superior: m ───────────────────────────────────────────────────────
ax_m.fill_between(x,
    bin_df["m_lo_boot"], bin_df["m_hi_boot"],
    color=COL_BLUE_FILL, alpha=0.70, zorder=2, label="95% bootstrap CI")
ax_m.plot(x, bin_df["m"],
    "o-", color=COL_BLUE, lw=2, markersize=5, zorder=3,
    label=r"$m = P(U) - P(C)$")
ax_m.axhline(0, linestyle="--", color="grey", alpha=0.55, lw=1, zorder=2)

ax_m.annotate(
    fr"$m=0$ at {x_m0:.1f}°C",
    xy=(x_m0, 0.01), xytext=(x_m0 + 0.8, 0.22),
    fontsize=8, color=COL_DARK, zorder=5,
    arrowprops=dict(arrowstyle="-", color=COL_DARK, lw=0.7)
)

if np.isfinite(transition_lo) and np.isfinite(transition_hi):
    ax_m.text(
        (transition_lo + transition_hi) / 2,
        ax_m.get_ylim()[0] + 0.03,
        fr"[{transition_lo:.1f}, {transition_hi:.1f}]°C",
        ha="center", va="bottom", fontsize=7.5,
        color=COL_DARK, alpha=0.70, zorder=5
    )

ax_m.set_ylabel(r"$m$", fontsize=12, color=COL_DARK)
ax_m.legend(fontsize=8, loc="upper left", frameon=True, framealpha=0.85)

# ── Panell inferior: q ───────────────────────────────────────────────────────
ax_q.fill_between(x,
    bin_df["q_lo_boot"], bin_df["q_hi_boot"],
    color=COL_ORG_FILL, alpha=0.70, zorder=2, label="95% bootstrap CI")
ax_q.plot(x, bin_df["q"],
    "s-", color=COL_ORANGE, lw=2, markersize=5, zorder=3,
    label=r"$q = P(U) + P(C)$")

q_mean = np.average(bin_df["q"], weights=bin_df["n"])
ax_q.axhline(q_mean, linestyle="--", color=COL_ORANGE, alpha=0.50, lw=1.2,
             label=fr"$\bar{{q}} \approx {q_mean:.2f}$")

ax_q.set_ylabel(r"$q$", fontsize=12, color=COL_DARK)
ax_q.set_xlabel(
    r"Corrected air temperature $T_{\mathrm{corr}}$ (°C)",
    fontsize=11, color=COL_DARK)
ax_q.set_ylim(
    max(0.0, bin_df["q_lo_boot"].min() - 0.06),
    min(1.0, bin_df["q_hi_boot"].max() + 0.06)
)
ax_q.legend(fontsize=8, loc="lower right", frameon=True, framealpha=0.85)

# ── Títol i guardar ──────────────────────────────────────────────────────────
fig.suptitle("Blume–Capel order parameters",
             fontsize=13, color=COL_DARK, y=1.01)
plt.tight_layout()
plt.savefig("F_bc_mq.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# ====================================================
# PLOT 1: m and q — two stacked subplots, same y-scale
# ====================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ── Variables del resultat ────────────────────────────────────────────────────
bin_df        = res_T["bin_df"]
x             = bin_df["x_mean"].values
x_m0          = res_T["x_m0"]
transition_lo = res_T["transition_lo"]
transition_hi = res_T["transition_hi"]

# ── Paleta ───────────────────────────────────────────────────────────────────
COL_AQUA      = "#9CE5E8"
COL_BLUE      = "#2C7AB5"
COL_DARK      = "#152F61"
COL_ORANGE    = "#E07B39"
COL_BLUE_FILL = "#C5DCF0"
COL_ORG_FILL  = "#F5D0B5"

# ── Rangs i alçades proporcionals ────────────────────────────────────────────
# ── Rangs i alçades proporcionals ────────────────────────────────────────────
m_lo = min(bin_df["m_lo_boot"].min(), -0.05)
m_hi = max(bin_df["m_hi_boot"].max(),  0.05)
q_lo = max(0.0, bin_df["q_lo_boot"].min() - 0.04)
q_hi = min(1.0, bin_df["q_hi_boot"].max() + 0.04)

m_span = m_hi - m_lo
q_span = q_hi - q_lo

fig, (ax_m, ax_q) = plt.subplots(
    2, 1,
    figsize=(7, 7),          # ← alçada total fixa, raonable
    sharex=True,
    gridspec_kw={
        "hspace": 0.08,
        "height_ratios": [m_span, q_span]   # ← proporcions correctes
    }
)

ax_m.set_ylim(m_lo, m_hi)
ax_q.set_ylim(q_lo, q_hi)

# ── Elements comuns ───────────────────────────────────────────────────────────
for ax in (ax_m, ax_q):
    if np.isfinite(transition_lo) and np.isfinite(transition_hi):
        ax.axvspan(transition_lo, transition_hi,
                   color=COL_AQUA, alpha=0.40, zorder=0)
    ax.axvline(x_m0, linestyle=":", color=COL_DARK,
               alpha=0.50, lw=1.2, zorder=1)
    ax.grid(True, alpha=0.18, zorder=0)
    ax.tick_params(axis="both", colors=COL_DARK, labelsize=10)
    for spine in ax.spines.values():
        spine.set_edgecolor(COL_DARK)
        spine.set_linewidth(0.8)

for ax in (ax_m, ax_q):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ── Panell superior: m ───────────────────────────────────────────────────────
ax_m.fill_between(x,
    bin_df["m_lo_boot"], bin_df["m_hi_boot"],
    color=COL_BLUE_FILL, alpha=0.70, zorder=2)
ax_m.plot(x, bin_df["m"],
    "o-", color=COL_BLUE, lw=2, markersize=5, zorder=3)
ax_m.axhline(0, linestyle="--", color="grey", alpha=0.55, lw=1, zorder=2)

ax_m.set_ylabel(r"$m = P(U) - P(C)$", fontsize=14, color=COL_DARK)

legend_m = [
    Line2D([0], [0], color=COL_BLUE, lw=2, marker="o", markersize=5,
           label=r"$m = P(U) - P(C)$"),
    Patch(facecolor=COL_BLUE_FILL, alpha=0.70,
          label="95% bootstrap CI"),
    Patch(facecolor=COL_AQUA, alpha=0.40,
          label=fr"crossover band [{transition_lo:.1f}, {transition_hi:.1f}]°C"),
    Line2D([0], [0], linestyle=":", color=COL_DARK, alpha=0.50, lw=1.2,
           label=fr"$m=0$ at {x_m0:.1f}°C"),
]
ax_m.legend(handles=legend_m, fontsize=11, loc="upper left",
            frameon=False, framealpha=0.90)

# ── Panell inferior: q ───────────────────────────────────────────────────────
ax_q.fill_between(x,
    bin_df["q_lo_boot"], bin_df["q_hi_boot"],
    color=COL_ORG_FILL, alpha=0.70, zorder=2)
ax_q.plot(x, bin_df["q"],
    "s-", color=COL_ORANGE, lw=2, markersize=5, zorder=3)

ax_q.set_ylabel(r"$q = P(U) + P(C)$", fontsize=14, color=COL_DARK)
ax_q.set_xlabel(
    r"$T_{\mathrm{corr}}$ (°C)",
    fontsize=14, color=COL_DARK)

legend_q = [
    Line2D([0], [0], color=COL_ORANGE, lw=2, marker="s", markersize=5,
           label=r"$q = P(U) + P(C)$"),
    Patch(facecolor=COL_ORG_FILL, alpha=0.70,
          label="95% bootstrap CI"),
]
ax_q.legend(handles=legend_q, fontsize=11, loc="upper left",
            frameon=False, framealpha=0.90)


plt.tight_layout()
plt.savefig("F_bc_mq.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ── Variables del resultat ────────────────────────────────────────────────────
bin_df        = res_T["bin_df"]
x             = bin_df["x_mean"].values
x_m0          = res_T["x_m0"]
transition_lo = res_T["transition_lo"]
transition_hi = res_T["transition_hi"]

REGIME_BOUNDS = [26.10, 30.77]

# ── Paleta ───────────────────────────────────────────────────────────────────
COL_AQUA      = "#9CE5E8"
COL_BLUE      = "#2C7AB5"
COL_DARK      = "#152F61"
COL_ORANGE    = "#E07B39"
COL_BLUE_FILL = "#C5DCF0"
COL_ORG_FILL  = "#F5D0B5"
COL_REGIME    = "#6B6B6B"

# ── Rangs i alçades proporcionals ────────────────────────────────────────────
m_lo = min(bin_df["m_lo_boot"].min(), -0.05)
m_hi = max(bin_df["m_hi_boot"].max(),  0.05)
q_lo = max(0.0, bin_df["q_lo_boot"].min() - 0.04)
q_hi = min(1.0, bin_df["q_hi_boot"].max() + 0.04)

m_span = m_hi - m_lo
q_span = q_hi - q_lo

fig, (ax_m, ax_q) = plt.subplots(
    2, 1,
    figsize=(7, 7),
    sharex=True,
    gridspec_kw={
        "hspace": 0.08,
        "height_ratios": [m_span, q_span]
    }
)

ax_m.set_ylim(m_lo, m_hi)
ax_q.set_ylim(q_lo, q_hi)

# ── Elements comuns ───────────────────────────────────────────────────────────
for ax in (ax_m, ax_q):
    if np.isfinite(transition_lo) and np.isfinite(transition_hi):
        ax.axvspan(transition_lo, transition_hi,
                   color=COL_AQUA, alpha=0.40, zorder=0)
    ax.axvline(x_m0, linestyle=":", color=COL_DARK,
               alpha=0.50, lw=1.2, zorder=1)
    for rb in REGIME_BOUNDS:
        ax.axvline(rb, linestyle="--", color=COL_REGIME,
                   alpha=0.85, lw=1.5, zorder=1)
    ax.grid(True, alpha=0.18, zorder=0)
    ax.tick_params(axis="both", colors=COL_DARK, labelsize=10)
    for spine in ax.spines.values():
        spine.set_edgecolor(COL_DARK)
        spine.set_linewidth(0.8)

for ax in (ax_m, ax_q):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ── Panell superior: m ───────────────────────────────────────────────────────
ax_m.fill_between(x,
    bin_df["m_lo_boot"], bin_df["m_hi_boot"],
    color=COL_BLUE_FILL, alpha=0.70, zorder=2)
ax_m.plot(x, bin_df["m"],
    "o-", color=COL_BLUE, lw=2, markersize=5, zorder=3)
ax_m.axhline(0, linestyle="--", color="grey", alpha=0.55, lw=1, zorder=2)

ax_m.set_ylabel(r"$m = P(U) - P(C)$", fontsize=14, color=COL_DARK)

legend_m = [
    Line2D([0], [0], color=COL_BLUE, lw=2, marker="o", markersize=5,
           label=r"$m = P(U) - P(C)$"),
    Patch(facecolor=COL_BLUE_FILL, alpha=0.70,
          label="95% bootstrap CI"),
    Patch(facecolor=COL_AQUA, alpha=0.40,
          label=fr"crossover band [{transition_lo:.1f}, {transition_hi:.1f}]°C"),
    Line2D([0], [0], linestyle="--", color=COL_REGIME, alpha=0.85, lw=1.5,
       label=fr"regime boundaries [{REGIME_BOUNDS[0]}, {REGIME_BOUNDS[1]}]°C"),
]
ax_m.legend(handles=legend_m, fontsize=11, loc="upper left",
            frameon=False, framealpha=0.90)

# ── Panell inferior: q ───────────────────────────────────────────────────────
ax_q.fill_between(x,
    bin_df["q_lo_boot"], bin_df["q_hi_boot"],
    color=COL_ORG_FILL, alpha=0.70, zorder=2)
ax_q.plot(x, bin_df["q"],
    "s-", color=COL_ORANGE, lw=2, markersize=5, zorder=3)

ax_q.set_ylabel(r"$q = P(U) + P(C)$", fontsize=14, color=COL_DARK)
ax_q.set_xlabel(r"$T_{\mathrm{corr}}$ (°C)", fontsize=14, color=COL_DARK)

legend_q = [
    Line2D([0], [0], color=COL_ORANGE, lw=2, marker="s", markersize=5,
           label=r"$q = P(U) + P(C)$"),
    Patch(facecolor=COL_ORG_FILL, alpha=0.70,
          label="95% bootstrap CI"),
]
ax_q.legend(handles=legend_q, fontsize=11, loc="upper left",
            frameon=False, framealpha=0.90)

plt.tight_layout()
plt.savefig("F_bc_mq.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def plot_bc_fields_paperstyle(
    res,
    output_pdf="F_bc_fields.pdf",
    output_png="F_bc_fields.png",
    x_label=r"$T_{\mathrm{corr}}$ (°C)",
    regime_bounds=(26.10, 30.77),
):
    """
    Paper-style plot of the effective single-site Blume-Capel fields:

        beta h = 1/2 log[P(U)/P(C)]
        beta D = -1/2 log[P(U)P(C)/P(N)^2]

    Requires a result dictionary returned by empirical_blume_capel(),
    containing bin_df with:
        x_mean,
        beta_h, beta_h_lo_boot, beta_h_hi_boot,
        beta_D, beta_D_lo_boot, beta_D_hi_boot
    """

    bin_df = res["bin_df"].copy().sort_values("x_mean")

    x = bin_df["x_mean"].values
    x_m0 = res.get("x_m0", np.nan)
    transition_lo = res.get("transition_lo", np.nan)
    transition_hi = res.get("transition_hi", np.nan)

    # --------------------------------------------------------
    # Palette
    # --------------------------------------------------------
    COL_AQUA      = "#9CE5E8"
    COL_BLUE      = "#2C7AB5"
    COL_DARK      = "#152F61"
    COL_ORANGE    = "#E07B39"
    COL_BLUE_FILL = "#C5DCF0"
    COL_ORG_FILL  = "#F5D0B5"
    COL_REGIME    = "#6B6B6B"

    # --------------------------------------------------------
    # Axis ranges and proportional heights
    # --------------------------------------------------------
    bh_lo = bin_df["beta_h_lo_boot"].min()
    bh_hi = bin_df["beta_h_hi_boot"].max()

    bD_lo = bin_df["beta_D_lo_boot"].min()
    bD_hi = bin_df["beta_D_hi_boot"].max()

    # Add small padding
    bh_pad = 0.08 * (bh_hi - bh_lo) if np.isfinite(bh_hi - bh_lo) else 0.1
    bD_pad = 0.08 * (bD_hi - bD_lo) if np.isfinite(bD_hi - bD_lo) else 0.1

    bh_lo = bh_lo - bh_pad
    bh_hi = bh_hi + bh_pad
    bD_lo = bD_lo - bD_pad
    bD_hi = bD_hi + bD_pad

    # Make sure zero is visible in beta h
    bh_lo = min(bh_lo, -0.05)
    bh_hi = max(bh_hi,  0.05)

    bh_span = bh_hi - bh_lo
    bD_span = bD_hi - bD_lo

    fig, (ax_h, ax_D) = plt.subplots(
        2, 1,
        figsize=(7, 7),
        sharex=True,
        gridspec_kw={
            "hspace": 0.08,
            "height_ratios": [bh_span, bD_span],
        }
    )

    ax_h.set_ylim(bh_lo, bh_hi)
    ax_D.set_ylim(bD_lo, bD_hi)

    # --------------------------------------------------------
    # Common elements
    # --------------------------------------------------------
    for ax in (ax_h, ax_D):

        if np.isfinite(transition_lo) and np.isfinite(transition_hi):
            ax.axvspan(
                transition_lo,
                transition_hi,
                color=COL_AQUA,
                alpha=0.40,
                zorder=0,
            )

        if np.isfinite(x_m0):
            ax.axvline(
                x_m0,
                linestyle=":",
                color=COL_DARK,
                alpha=0.50,
                lw=1.2,
                zorder=1,
            )

        for rb in regime_bounds:
            ax.axvline(
                rb,
                linestyle="--",
                color=COL_REGIME,
                alpha=0.85,
                lw=1.5,
                zorder=1,
            )

        ax.grid(True, alpha=0.18, zorder=0)
        ax.tick_params(axis="both", colors=COL_DARK, labelsize=10)

        for spine in ax.spines.values():
            spine.set_edgecolor(COL_DARK)
            spine.set_linewidth(0.8)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # --------------------------------------------------------
    # Top panel: beta h
    # --------------------------------------------------------
    ax_h.fill_between(
        x,
        bin_df["beta_h_lo_boot"],
        bin_df["beta_h_hi_boot"],
        color=COL_BLUE_FILL,
        alpha=0.70,
        zorder=2,
    )

    ax_h.plot(
        x,
        bin_df["beta_h"],
        "o-",
        color=COL_BLUE,
        lw=2,
        markersize=5,
        zorder=3,
    )

    ax_h.axhline(
        0,
        linestyle="--",
        color="grey",
        alpha=0.55,
        lw=1,
        zorder=2,
    )

    ax_h.set_ylabel(r"$\beta h$", fontsize=14, color=COL_DARK)

    legend_h = [
        Line2D(
            [0], [0],
            color=COL_BLUE,
            lw=2,
            marker="o",
            markersize=5,
            label=r"$\beta h = \frac{1}{2}\ln\frac{P(U)}{P(C)}$",
        ),
        Patch(
            facecolor=COL_BLUE_FILL,
            alpha=0.70,
            label="95% bootstrap CI",
        ),
    ]

    if np.isfinite(transition_lo) and np.isfinite(transition_hi):
        legend_h.append(
            Patch(
                facecolor=COL_AQUA,
                alpha=0.40,
                label=fr"crossover band [{transition_lo:.1f}, {transition_hi:.1f}]°C",
            )
        )

    legend_h.append(
        Line2D(
            [0], [0],
            linestyle="--",
            color=COL_REGIME,
            alpha=0.85,
            lw=1.5,
            label=fr"regime boundaries [{regime_bounds[0]}, {regime_bounds[1]}]°C",
        )
    )

    ax_h.legend(
        handles=legend_h,
        fontsize=10,
        loc="upper left",
        frameon=False,
        framealpha=0.90,
    )

    # --------------------------------------------------------
    # Bottom panel: beta D
    # --------------------------------------------------------
    ax_D.fill_between(
        x,
        bin_df["beta_D_lo_boot"],
        bin_df["beta_D_hi_boot"],
        color=COL_ORG_FILL,
        alpha=0.70,
        zorder=2,
    )

    ax_D.plot(
        x,
        bin_df["beta_D"],
        "s-",
        color=COL_ORANGE,
        lw=2,
        markersize=5,
        zorder=3,
    )

    ax_D.axhline(
        0,
        linestyle="--",
        color="grey",
        alpha=0.55,
        lw=1,
        zorder=2,
    )

    ax_D.set_ylabel(r"$\beta D$", fontsize=14, color=COL_DARK)
    ax_D.set_xlabel(x_label, fontsize=14, color=COL_DARK)

    legend_D = [
        Line2D(
            [0], [0],
            color=COL_ORANGE,
            lw=2,
            marker="s",
            markersize=5,
            label=r"$\beta D = -\frac{1}{2}\ln\frac{P(U)P(C)}{P(N)^2}$",
        ),
        Patch(
            facecolor=COL_ORG_FILL,
            alpha=0.70,
            label="95% bootstrap CI",
        ),
    ]

    ax_D.legend(
        handles=legend_D,
        fontsize=10,
        loc="lower left",
        frameon=False,
        framealpha=0.90,
    )

    plt.tight_layout()

    fig.savefig(output_pdf, bbox_inches="tight", dpi=300)
    fig.savefig(output_png, bbox_inches="tight", dpi=300)

    plt.show()

    return fig

In [ ]:
fig = plot_bc_fields_paperstyle(
    res_T,
    output_pdf="F_bc_fields.pdf",
    output_png="F_bc_fields.png",
    x_label=r"$T_{\mathrm{corr}}$ (°C)",
    regime_bounds=(26.10, 30.77),
)